In [ ]:
import sys
print(sys.version)

In [ ]:
import os
import tensorflow as tf

# Make only GPU 7 visible to TensorFlow
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# List the visible GPUs (GPU 7 will appear as GPU 0 to TensorFlow)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Allow memory growth instead of setting a fixed limit
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f"Using GPU 3 (seen as GPU 0 by TF): {len(gpus)} physical, {len(logical_gpus)} logical GPUs")
    except RuntimeError as e:
        print("TF GPU setup error:", e)
else:
    print("No GPU available. Running on CPU.")

In [ ]:
import os
import cv2
import numpy as np
import pywt
import scipy.stats
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from skimage.feature import graycomatrix, graycoprops
from tensorflow.keras.utils import to_categorical
import umap
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.tree import DecisionTreeClassifier, _tree, plot_tree
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import itertools
import tqdm
from transformers import TFViTModel, ViTFeatureExtractor


In [ ]:
# --- CONFIGURATION ---
IMG_SIZE = (224, 224)
WAVELET = 'db1'

TRAIN_DIR = './MES Mixed Data' # <-- Change this to your training dataset path
TEST_DIR  = './MES classification_20250313' # <-- Change this to your testing dataset path
experiment_saved_name = "TrainingMixedTesting1"

# --- WAVELET + GLCM FEATURE EXTRACTORS ---
def extract_wavelet_stats(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    coeffs2 = pywt.dwt2(gray, WAVELET)
    LL, (LH, HL, HH) = coeffs2
    def stats(subband):
        return [
            np.mean(subband),
            np.std(subband),
            np.var(subband),
            scipy.stats.entropy(np.abs(subband.flatten()) + 1e-6)
        ]
    features = []
    for band in [LL, LH, HL, HH]:
        features.extend(stats(band))
    hh_energy = np.sum(np.square(HH))
    features.append(hh_energy)
    return features

def extract_glcm_features_extended(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    angles = [0, np.pi/4, np.pi/2]
    glcm = graycomatrix(gray, distances=[5], angles=angles, levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').mean()
    dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    return [contrast, dissimilarity, homogeneity]

# --- REUSABLE DATA LOADER ---
def load_dataset(folder_path):
    X_img, X_feat, y_label, img_paths = [], [], [], []
    for label in os.listdir(folder_path):
        label_path = os.path.join(folder_path, label)
        if not os.path.isdir(label_path): continue
        for fname in os.listdir(label_path):
            img_path = os.path.join(label_path, fname)
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            cropped = img[30:430, 200:550]
            resized = cv2.resize(cropped, IMG_SIZE)
            wavelet_feats = extract_wavelet_stats(resized)
            glcm_feats = extract_glcm_features_extended(resized)
            combined = wavelet_feats + glcm_feats
            X_img.append(resized)
            X_feat.append(combined)
            y_label.append(label)
            img_paths.append(img_path)
    return np.array(X_img), np.array(X_feat), np.array(y_label), np.array(img_paths)

# --- LOAD TRAINING + TEST SET ---
X_img_train_raw, X_feat_train_raw, y_train_label, img_paths_train = load_dataset(TRAIN_DIR)
X_img_test_raw,  X_feat_test_raw,  y_test_label,  img_paths_test  = load_dataset(TEST_DIR)

# --- NORMALIZE IMAGES ---
X_img_train = X_img_train_raw.astype(np.float32) / 255.0
X_img_test  = X_img_test_raw.astype(np.float32) / 255.0

# --- LABEL ENCODING ---
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_label)
y_test_encoded  = le.transform(y_test_label)
y_train_cat = to_categorical(y_train_encoded, num_classes=len(le.classes_))
y_test_cat  = to_categorical(y_test_encoded,  num_classes=len(le.classes_))

# --- FEATURE SCALING ---
scaler = StandardScaler()
X_feat_train_scaled = scaler.fit_transform(X_feat_train_raw)
X_feat_test_scaled  = scaler.transform(X_feat_test_raw)

# --- PLOT TRAIN CLASS DISTRIBUTION (Before SMOTE) ---
plt.figure(figsize=(8, 4))
plt.title("Training Set Class Distribution (Before SMOTE)")
plt.bar(*np.unique(y_train_encoded, return_counts=True), tick_label=le.classes_)
plt.xlabel("Class")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.show()

# --- PLOT TEST CLASS DISTRIBUTION ---
plt.figure(figsize=(8, 4))
plt.title("Testing Set Class Distribution")
plt.bar(*np.unique(y_test_encoded, return_counts=True), tick_label=le.classes_, color='orange')
plt.xlabel("Class")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.show()

# --- APPLY SMOTE TO TRAINING SET ---
smote = SMOTE(random_state=42)
X_feat_train_bal, y_train_bal = smote.fit_resample(X_feat_train_scaled, y_train_encoded)

# --- PLOT TRAIN CLASS DISTRIBUTION (After SMOTE) ---
plt.figure(figsize=(8, 4))
plt.title("Training Set Class Distribution (After SMOTE)")
plt.bar(*np.unique(y_train_bal, return_counts=True), tick_label=le.classes_, color='green')
plt.xlabel("Class")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.show()

# --- MAP BALANCED FEATURES TO REAL IMAGES ---
X_img_train_bal, img_paths_train_bal = [], []
for feat, label in zip(X_feat_train_bal, y_train_bal):
    dists = np.linalg.norm(X_feat_train_scaled[y_train_encoded == label] - feat, axis=1)
    idx = np.where(y_train_encoded == label)[0][np.argmin(dists)]
    X_img_train_bal.append(X_img_train[idx])
    img_paths_train_bal.append(img_paths_train[idx])
X_img_train_bal = np.array(X_img_train_bal, dtype=np.float32)
y_train_cat_bal = to_categorical(y_train_bal, num_classes=len(le.classes_))

# --- UMAP PROJECTION (fit on training, apply to test) ---
umap_reducer = umap.UMAP(n_neighbors=10, min_dist=0.05, n_components=2, metric='euclidean', random_state=42)
X_train_umap = umap_reducer.fit_transform(X_feat_train_bal)
X_test_umap  = umap_reducer.transform(X_feat_test_scaled)

# --- SHAPE SUMMARY ---
print(f"X_img_train_bal: {X_img_train_bal.shape}, X_img_test: {X_img_test.shape}")
print(f"X_feat_train_bal: {X_feat_train_bal.shape}, X_feat_test_scaled: {X_feat_test_scaled.shape}")
print(f"X_train_umap: {X_train_umap.shape}, X_test_umap: {X_test_umap.shape}")
print(f"y_train_cat_bal: {y_train_cat_bal.shape}, y_test_cat: {y_test_cat.shape}")


In [ ]:
import matplotlib.pyplot as plt

# --- Show training example ---
img_train = cv2.imread(img_paths_train[0])
img_train = cv2.cvtColor(img_train, cv2.COLOR_BGR2RGB)
cropped_train = img_train[30:430, 200:550]
resized_train = cv2.resize(cropped_train, IMG_SIZE)

# --- Show testing example ---
img_test = cv2.imread(img_paths_test[0])
img_test = cv2.cvtColor(img_test, cv2.COLOR_BGR2RGB)
cropped_test = img_test[30:430, 200:550]
resized_test = cv2.resize(cropped_test, IMG_SIZE)

# --- Plot side-by-side comparison ---
fig, axs = plt.subplots(2, 3, figsize=(12, 6))

# Row 1: Training example
axs[0, 0].imshow(img_train)
axs[0, 0].set_title("Train - Original")
axs[0, 1].imshow(cropped_train)
axs[0, 1].set_title("Train - Cropped")
axs[0, 2].imshow(resized_train)
axs[0, 2].set_title("Train - Resized 224x224")

# Row 2: Testing example
axs[1, 0].imshow(img_test)
axs[1, 0].set_title("Test - Original")
axs[1, 1].imshow(cropped_test)
axs[1, 1].set_title("Test - Cropped")
axs[1, 2].imshow(resized_test)
axs[1, 2].set_title("Test - Resized 224x224")

# Formatting
for ax in axs.flatten():
    ax.axis('off')
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Rescale to 0–255 for visualization purposes ---
train_original = X_img_train * 255.0
test_original  = X_img_test * 255.0

# --- Flatten pixel values across all channels and images ---
train_pixels_orig = train_original.flatten()
test_pixels_orig  = test_original.flatten()

train_pixels_norm = X_img_train.flatten()
test_pixels_norm  = X_img_test.flatten()

# --- Plot original pixel distributions (0–255) ---
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(train_pixels_orig, bins=50, color='gray', alpha=0.7, label='Train')
plt.hist(test_pixels_orig,  bins=50, color='red', alpha=0.5, label='Test')
plt.title("Pixel Value Distribution [0–255]")
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.legend()

# --- Plot normalized pixel distributions (0–1) ---
plt.subplot(1, 2, 2)
plt.hist(train_pixels_norm, bins=50, color='blue', alpha=0.7, label='Train')
plt.hist(test_pixels_norm,  bins=50, color='orange', alpha=0.5, label='Test')
plt.title("Pixel Value Distribution After Normalization [0–1]")
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Use handcrafted feature names
feature_names = [
    'LL_mean', 'LL_std', 'LL_var', 'LL_entropy',
    'LH_mean', 'LH_std', 'LH_var', 'LH_entropy',
    'HL_mean', 'HL_std', 'HL_var', 'HL_entropy',
    'HH_mean', 'HH_std', 'HH_var', 'HH_entropy',
    'GLCM_contrast', 'GLCM_dissimilarity', 'GLCM_homogeneity',
    'HH_energy'
]

# Create raw DataFrames
df_train_raw = pd.DataFrame(X_feat_train_raw[:, :20], columns=feature_names)
df_test_raw  = pd.DataFrame(X_feat_test_raw[:, :20],  columns=feature_names)

# Standardize using same scaler
scaler = StandardScaler()
df_train_scaled = pd.DataFrame(scaler.fit_transform(df_train_raw), columns=feature_names)
df_test_scaled  = pd.DataFrame(scaler.transform(df_test_raw),  columns=feature_names)

# Plot 1: Train - Before Scaling
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_train_raw)
plt.title("Train Set - Handcrafted Features (Before Standardization)")
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 2: Train - After Scaling
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_train_scaled)
plt.title("Train Set - Handcrafted Features (After Standardization)")
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 3: Test - Before Scaling
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_test_raw)
plt.title("Test Set - Handcrafted Features (Before Standardization)")
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 4: Test - After Scaling
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_test_scaled)
plt.title("Test Set - Handcrafted Features (After Standardization)")
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Handcrafted feature names
feature_names = [
    'LL_mean', 'LL_std', 'LL_var', 'LL_entropy',
    'LH_mean', 'LH_std', 'LH_var', 'LH_entropy',
    'HL_mean', 'HL_std', 'HL_var', 'HL_entropy',
    'HH_mean', 'HH_std', 'HH_var', 'HH_entropy',
    'HH_energy', 'GLCM_contrast', 'GLCM_dissimilarity', 'GLCM_homogeneity'
]

# Prepare raw DataFrames
X_feats_train = np.array(X_feat_train_raw, dtype=np.float32)
X_feats_test  = np.array(X_feat_test_raw, dtype=np.float32)
df_train_raw = pd.DataFrame(X_feats_train, columns=feature_names)
df_test_raw  = pd.DataFrame(X_feats_test,  columns=feature_names)

# Standardize using same scaler (like in real pipeline)
scaler = StandardScaler()
X_feats_train_scaled = scaler.fit_transform(X_feats_train)
X_feats_test_scaled  = scaler.transform(X_feats_test)
df_train_scaled = pd.DataFrame(X_feats_train_scaled, columns=feature_names)
df_test_scaled  = pd.DataFrame(X_feats_test_scaled,  columns=feature_names)

# --- PLOT 1: Train - Before Scaling (Log Scale) ---
plt.figure(figsize=(15, 5))
sns.boxplot(data=df_train_raw, palette="pastel")
plt.yscale('log')
plt.title("Train - Handcrafted Features Before Standardization (Log Scale)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# --- PLOT 2: Train - After Scaling ---
plt.figure(figsize=(15, 5))
sns.boxplot(data=df_train_scaled, palette="husl")
plt.title("Train - Handcrafted Features After Standardization")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# --- PLOT 3: Test - Before Scaling (Log Scale) ---
plt.figure(figsize=(15, 5))
sns.boxplot(data=df_test_raw, palette="pastel")
plt.yscale('log')
plt.title("Test - Handcrafted Features Before Standardization (Log Scale)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# --- PLOT 4: Test - After Scaling ---
plt.figure(figsize=(15, 5))
sns.boxplot(data=df_test_scaled, palette="husl")
plt.title("Test - Handcrafted Features After Standardization")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# --- MOD-SE(2) CNN (UNCHANGED) ---
def z2_se2n(input_tensor, kernel, orientations_nb, periodicity=2 * np.pi, diskMask=True, padding='VALID'):
    print("Base Kernel:\n", kernel.numpy())
    kernel_stack = rotate_lifting_kernels(kernel, orientations_nb, periodicity=periodicity, diskMask=diskMask)
    print("Z2-SE2N ROTATED KERNEL SET SHAPE:", kernel_stack.get_shape())
    kernels_as_if_2D = tf.transpose(kernel_stack, [1, 2, 3, 0, 4])
    kernelSizeH, kernelSizeW, channelsIN, channelsOUT = map(int, kernel.shape)
    kernels_as_if_2D = tf.reshape(kernels_as_if_2D, [kernelSizeH, kernelSizeW, channelsIN, orientations_nb * channelsOUT])
    layer_output = tf.nn.conv2d(input=input_tensor, filters=kernels_as_if_2D, strides=[1, 1, 1, 1], padding=padding)
    layer_output = tf.reshape(layer_output, [tf.shape(layer_output)[0], int(layer_output.shape[1]), int(layer_output.shape[2]), orientations_nb, channelsOUT])
    print("OUTPUT SE2N ACTIVATIONS SHAPE:", layer_output.get_shape())
    return layer_output, kernel_stack

def se2n_se2n(input_tensor, kernel, periodicity=2 * np.pi, diskMask=True, padding='VALID'):
    kernelSizeH, kernelSizeW, orientations_nb, channelsIN, channelsOUT = map(int, kernel.shape)
    kernel_stack = rotate_gconv_kernels(kernel, periodicity, diskMask)
    print("SE2N-SE2N ROTATED KERNEL SET SHAPE:", kernel_stack.get_shape())
    input_tensor_as_if_2D = tf.reshape(input_tensor, [tf.shape(input_tensor)[0], int(input_tensor.shape[1]), int(input_tensor.shape[2]), orientations_nb * channelsIN])
    kernels_as_if_2D = tf.transpose(kernel_stack, [1, 2, 3, 4, 0, 5])
    kernels_as_if_2D = tf.reshape(kernels_as_if_2D, [kernelSizeH, kernelSizeW, orientations_nb * channelsIN, orientations_nb * channelsOUT])
    layer_output = tf.nn.conv2d(input=input_tensor_as_if_2D, filters=kernels_as_if_2D, strides=[1, 1, 1, 1], padding=padding)
    layer_output = tf.reshape(layer_output, [tf.shape(layer_output)[0], int(layer_output.shape[1]), int(layer_output.shape[2]), orientations_nb, channelsOUT])
    print("OUTPUT SE2N ACTIVATIONS SHAPE:", layer_output.get_shape())
    return layer_output, kernel_stack

def spatial_max_pool(input_tensor, nbOrientations, padding='SAME'):
    activations = [None] * nbOrientations
    for i in range(nbOrientations):
        activations[i] = tf.nn.max_pool(value=input_tensor[:, :, :, i, :], ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding=padding)
    tensor_pooled = tf.stack(activations, axis=-1)
    return tensor_pooled

def rotate_lifting_kernels(kernel, orientations_nb, periodicity=2 * np.pi, diskMask=True):
    kernelSizeH, kernelSizeW, channelsIN, channelsOUT = map(int, kernel.shape)
    print("Z2-SE2N BASE KERNEL SHAPE:", kernel.get_shape())
    kernel_flat = tf.reshape(kernel, [kernelSizeH * kernelSizeW, channelsIN * channelsOUT])
    idx, vals = MultiRotationOperatorMatrixSparse([kernelSizeH, kernelSizeW], orientations_nb, periodicity=periodicity, diskMask=diskMask)
    rotOp_matrix = tf.SparseTensor(idx, vals, [orientations_nb * kernelSizeH * kernelSizeW, kernelSizeH * kernelSizeW])
    set_of_rotated_kernels = tf.sparse_tensor_dense_matmul(rotOp_matrix, kernel_flat)
    set_of_rotated_kernels = tf.reshape(set_of_rotated_kernels, [orientations_nb, kernelSizeH, kernelSizeW, channelsIN, channelsOUT])
    return set_of_rotated_kernels

def rotate_gconv_kernels(kernel, periodicity=2 * np.pi, diskMask=True):
    kernelSizeH, kernelSizeW, orientations_nb, channelsIN, channelsOUT = map(int, kernel.shape)
    print("SE2N-SE2N BASE KERNEL SHAPE:", kernel.get_shape())
    kernel_flat = tf.reshape(kernel, [kernelSizeH * kernelSizeW, orientations_nb * channelsIN * channelsOUT])
    idx, vals = MultiRotationOperatorMatrixSparse([kernelSizeH, kernelSizeW], orientations_nb, periodicity=periodicity, diskMask=diskMask)
    rotOp_matrix = tf.SparseTensor(idx, vals, [orientations_nb * kernelSizeH * kernelSizeW, kernelSizeH * kernelSizeW])
    kernels_planar_rotated = tf.sparse_tensor_dense_matmul(rotOp_matrix, kernel_flat)
    kernels_planar_rotated = tf.reshape(kernels_planar_rotated, [orientations_nb, kernelSizeH, kernelSizeW, orientations_nb, channelsIN, channelsOUT])
    set_of_rotated_kernels = [None] * orientations_nb
    for orientation in range(orientations_nb):
        kernels_temp = kernels_planar_rotated[orientation]
        kernels_temp = tf.transpose(kernels_temp, [0, 1, 3, 4, 2])
        kernels_temp = tf.reshape(kernels_temp, [kernelSizeH * kernelSizeW * channelsIN * channelsOUT, orientations_nb])
        roll_matrix = tf.constant(np.roll(np.identity(orientations_nb), orientation, axis=1), dtype=tf.float32)
        kernels_temp = tf.matmul(kernels_temp, roll_matrix)
        kernels_temp = tf.reshape(kernels_temp, [kernelSizeH, kernelSizeW, channelsIN, channelsOUT, orientations_nb])
        kernels_temp = tf.transpose(kernels_temp, [0, 1, 4, 2, 3])
        set_of_rotated_kernels[orientation] = kernels_temp
    return tf.stack(set_of_rotated_kernels)

def CoordRotationInv(ij, NiNj, theta):
    centeri = np.floor(NiNj[0] / 2)
    centerj = np.floor(NiNj[1] / 2)
    ijOld = np.zeros([2])
    ijOld[0] = np.cos(theta) * (ij[0] - centeri) + np.sin(theta) * (ij[1] - centerj) + centeri
    ijOld[1] = -np.sin(theta) * (ij[0] - centeri) + np.cos(theta) * (ij[1] - centerj) + centerj
    return ijOld

def LinIntIndicesAndWeights(ij, NiNj):
    i, j = ij
    Ni, Nj = NiNj
    i1 = int(np.floor(i))
    i2 = i1 + 1
    j1 = int(np.floor(j))
    j2 = j1 + 1
    ti = i - i1
    tj = j - j1
    w11 = (1 - ti) * (1 - tj)
    w12 = (1 - ti) * tj
    w21 = ti * (1 - tj)
    w22 = ti * tj
    indicesAndWeights = []
    if (0 <= i1 < Ni) and (0 <= j1 < Nj):
        indicesAndWeights.append([i1, j1, w11])
    if (0 <= i1 < Ni) and (0 <= j2 < Nj):
        indicesAndWeights.append([i1, j2, w12])
    if (0 <= i2 < Ni) and (0 <= j1 < Nj):
        indicesAndWeights.append([i2, j1, w21])
    if (0 <= i2 < Ni) and (0 <= j2 < Nj):
        indicesAndWeights.append([i2, j2, w22])
    return indicesAndWeights

def ToLinearIndex(ij, NiNj):
    return ij[0] * NiNj[0] + ij[1]

def RotationOperatorMatrix(NiNj, theta, diskMask=True):
    Ni, Nj = NiNj
    cij = np.floor(Ni / 2)
    rotationMatrix = np.zeros([Ni * Nj, Ni * Nj])
    for i in range(NiNj[0]):
        for j in range(NiNj[0]):
            if not(diskMask) or ((i - cij) * (i - cij) + (j - cij) * (j - cij) <= (cij + 0.5) * (cij + 0.5)):
                linij = ToLinearIndex([i, j], NiNj)
                ijOld = CoordRotationInv([i, j], NiNj, theta)
                linIntIndicesAndWeights = LinIntIndicesAndWeights(ijOld, NiNj)
                for indexAndWeight in linIntIndicesAndWeights:
                    indexOld = [indexAndWeight[0], indexAndWeight[1]]
                    linIndexOld = ToLinearIndex(indexOld, NiNj)
                    weight = indexAndWeight[2]
                    rotationMatrix[linij, linIndexOld] = weight
    return rotationMatrix

def MultiRotationOperatorMatrixSparse(NiNj, Ntheta, periodicity=2 * np.pi, diskMask=True):
    idx, vals = [], []
    for r in range(Ntheta):
        idxr, valsr = RotationOperatorMatrixSparse(NiNj, periodicity * r / Ntheta, linIndOffset=r * NiNj[0] * NiNj[1], diskMask=diskMask)
        idx += idxr
        vals += valsr
    return idx, vals

def RotationOperatorMatrixSparse(NiNj, theta, diskMask=True, linIndOffset=0):
    Ni, Nj = NiNj
    cij = np.floor(Ni / 2)
    idx, vals = [], []
    for i in range(NiNj[0]):
        for j in range(NiNj[0]):
            if not(diskMask) or ((i - cij) * (i - cij) + (j - cij) * (j - cij) <= (cij + 0.5) * (cij + 0.5)):
                linij = ToLinearIndex([i, j], NiNj)
                ijOld = CoordRotationInv([i, j], NiNj, theta)
                linIntIndicesAndWeights = LinIntIndicesAndWeights(ijOld, NiNj)
                for indexAndWeight in linIntIndicesAndWeights:
                    indexOld = [indexAndWeight[0], indexAndWeight[1]]
                    linIndexOld = ToLinearIndex(indexOld, NiNj)
                    weight = indexAndWeight[2]
                    idx.append((linij + linIndOffset, linIndexOld))
                    vals.append(weight)
    return tuple(idx), tuple(vals)

def GroupConv2D(filters, kernel_size, strides=(1, 1), padding='same', groups=3):
    def layer(x):
        group_list = []
        in_channels = x.shape[-1]
        assert in_channels % groups == 0, f"Number of input channels ({in_channels}) must be divisible by groups ({groups})"
        group_size = in_channels // groups
        for i in range(groups):
            x_group = x[:, :, :, i * group_size : (i + 1) * group_size]
            group_conv = tf.keras.layers.Conv2D(filters // groups, kernel_size, strides=strides, padding=padding)(x_group)
            group_list.append(group_conv)
        x = Concatenate()(group_list)
        x = BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        return x
    return layer

def SE2MaxPooling2D(pool_size=(2, 2)):
    def layer(x):
        x = tf.keras.layers.MaxPooling2D(pool_size=pool_size)(x)
        return x
    return layer

def SE2LiftingLayer(x):
    N, H, W, C = x.shape
    assert C % 3 == 0, "Number of input channels must be divisible by 3"
    group_size = C // 3
    x = tf.keras.layers.Reshape((H, W, 3, group_size))(x)
    x = tf.keras.layers.Permute((1, 2, 4, 3))(x)
    return x

def create_SE2CNN_model(input_shape, num_classes, dropout_rate=0.5):
    input_layer = Input(shape=input_shape)
    x = input_layer
    x = GroupConv2D(32, (3, 3))(x)
    x = SE2MaxPooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = GroupConv2D(64, (3, 3))(x)
    x = SE2MaxPooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = GroupConv2D(128, (3, 3))(x)
    x = SE2MaxPooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = GroupConv2D(256, (3, 3))(x)
    x = SE2MaxPooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = GroupConv2D(512, (3, 3))(x)
    x = SE2MaxPooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = GroupConv2D(1024, (3, 3))(x)
    x = SE2MaxPooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = SE2LiftingLayer(x)
    x = tf.keras.layers.Flatten()(x)
    x = Dense(1056, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    output = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=input_layer, outputs=output)
    return model

In [ ]:
from tensorflow.keras.layers import Lambda

# --- BUILD HYBRID MODEL ---
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from transformers import TFViTModel

def build_hybrid_model(image_input_shape, feat_input_shape, umap_feat_shape, num_classes, dropout_rate=0.4):
    # --- Mod-SE(2) CNN Branch (Unchanged) ---
    image_input_se2 = Input(shape=image_input_shape, name='image_input_se2')
    cnn_branch = create_SE2CNN_model(image_input_shape, num_classes, dropout_rate)
    x_se2 = cnn_branch(image_input_se2)
    x_se2 = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(x_se2)
    x_se2 = BatchNormalization()(x_se2)
    x_se2 = Dropout(dropout_rate)(x_se2)

    # --- Handcrafted Feature Branch ---
    feat_input = Input(shape=feat_input_shape, name='feat_input')
    x_feat = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(feat_input)
    x_feat = BatchNormalization()(x_feat)
    x_feat = Dropout(dropout_rate)(x_feat)

    # --- UMAP Feature Branch ---
    umap_input = Input(shape=umap_feat_shape, name='umap_feat_input')
    x_umap = Dense(32, activation='relu', kernel_regularizer=l2(0.01))(umap_input)
    x_umap = BatchNormalization()(x_umap)
    x_umap = Dropout(dropout_rate)(x_umap)

    # --- Fusion ---
    combined = Concatenate()([x_se2, x_feat, x_umap])
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(combined)
    x = Dropout(dropout_rate)(x)
    output = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=[image_input_se2, feat_input, umap_input], outputs=output)
    return model

In [ ]:
import tensorflow as tf

def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-8, 1.0)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.math.pow(1 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * cross_entropy, axis=1))
    return loss

In [ ]:
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ✅ Use balanced integer labels from SMOTE
y_train_labels = y_train_bal  # already integer-encoded

# ✅ Compute class weights for focal loss / training balance
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_labels), y=y_train_labels)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}

# ✅ Define training callbacks
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=10, verbose=1, mode='max')
]

# ✅ Build hybrid model (Mod-SE(2) + handcrafted + UMAP)
model_hybrid = build_hybrid_model(
    image_input_shape=(224, 224, 3),
    feat_input_shape=(20,),       # handcrafted
    umap_feat_shape=(2,),         # UMAP projection
    num_classes=4,
    dropout_rate=0.4
)

# ✅ Compile model with focal loss
model_hybrid.compile(
    optimizer=Adam(1e-5),
    loss=focal_loss(gamma=2.5, alpha=0.25),
    metrics=['accuracy']
)

# ✅ Optional: Use data augmentation on training images (not required if inputs are balanced already)
datagen = ImageDataGenerator(
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rotation_range=20,
    brightness_range=[0.8, 1.2],
    shear_range=0.2,
    fill_mode='nearest'
)
datagen.fit(X_img_train_bal)

# ✅ Set training and validation inputs
train_inputs = [X_img_train_bal, X_feat_train_bal, X_train_umap]
val_inputs = [X_img_test, X_feat_test_scaled, X_test_umap]

# ✅ Fit model
history = model_hybrid.fit(
    train_inputs, y_train_cat_bal,
    validation_data=(val_inputs, y_test_cat),
    batch_size=16,
    epochs=20,
    class_weight=class_weight_dict,
    #callbacks=callbacks,
    verbose=1
)


In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'], label='train_accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.legend()
plt.show()
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.show()

In [ ]:
from tensorflow.keras.utils import plot_model
plot_model(model_hybrid, to_file="model_3branch.architecture.png", show_shapes=True, show_layer_names=True)
model_hybrid.save(f"./Result/CrossDataset/{experiment_saved_name}/TryFindingBestModel.h5")

In [ ]:
# --- PARAMETER TUNING FOR DECISION TREE ---
max_depth_range = [3, 5, 7, 9, 11]
min_samples_leaf_range = [1, 3, 5, 10, 15]
threshold_range = np.linspace(0.3, 0.9, 13)

y_true = np.argmax(y_test_cat, axis=1)
y_pred_proba = model_hybrid.predict([X_img_test, X_feat_test_scaled, X_test_umap], verbose=0)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)

best_acc = 0
best_params = {}

for max_depth, min_samples_leaf in tqdm.tqdm(itertools.product(max_depth_range, min_samples_leaf_range)):
    tree = DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=min_samples_leaf, random_state=42)
    tree.fit(X_train_umap, y_train_labels)

    def generate_rule_function(tree):
        tree_ = tree.tree_
        def recurse(node):
            if tree_.feature[node] != _tree.TREE_UNDEFINED:
                idx = tree_.feature[node]
                thr = tree_.threshold[node]
                return f"(x[{idx}] <= {thr}) and ({recurse(tree_.children_left[node])}) or " \
                       f"(x[{idx}] > {thr}) and ({recurse(tree_.children_right[node])})"
            else:
                pred = np.argmax(tree_.value[node][0])
                return f"(return_val := {pred})"
        func_code = f"def rule_based_predict(x):\n    global return_val\n    {recurse(0)}\n    return return_val"
        return func_code

    exec(generate_rule_function(tree), globals())
    y_pred_rule = np.array([rule_based_predict(x) for x in X_test_umap])

    for threshold in threshold_range:
        y_pred_fused = np.where(np.max(y_pred_proba, axis=1) < threshold, y_pred_rule, y_pred_hybrid)
        acc = accuracy_score(y_true, y_pred_fused)
        if acc > best_acc:
            best_acc = acc
            best_params = {
                'max_depth': max_depth,
                'min_samples_leaf': min_samples_leaf,
                'threshold': threshold,
                'tree_model': tree,
                'final_prediction': y_pred_fused.copy()
            }

print(f"✅ Best Accuracy: {best_acc:.4f}")
print("Best Parameters:", {k: v for k, v in best_params.items() if k != 'final_prediction'})

In [ ]:
import joblib
import os
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# --- SAVE DIRECTORY ---
SAVE_DIR = f"./Result/CrossDataset/{experiment_saved_name}/"
os.makedirs(SAVE_DIR, exist_ok=True)

# --- FINAL DECISION TREE BASED ON BEST PARAMS ---
tree_umap = DecisionTreeClassifier(
    max_depth=best_params['max_depth'],
    min_samples_leaf=best_params['min_samples_leaf'],
    random_state=42
)
tree_umap.fit(X_train_umap, y_train_labels)

# --- SAVE MODELS ---
joblib.dump(tree_umap, os.path.join(SAVE_DIR, "rule_tree_mixed.pkl"))
joblib.dump(umap_reducer, os.path.join(SAVE_DIR, "umap_model_mixed.pkl"))
joblib.dump(scaler, os.path.join(SAVE_DIR, "scaler_handcrafted_20.pkl"))

# --- PLOT TREE ---
fig_width = 40 if tree_umap.get_depth() < 8 else 100
plt.figure(figsize=(fig_width, 20))
plot_tree(
    tree_umap,
    feature_names=[f"UMAP{i}" for i in range(X_train_umap.shape[1])],
    class_names=[f'MES {i}' for i in range(4)],
    filled=True,
    rounded=True,
    fontsize=12
)
plt.title(f"Best Decision Tree on UMAP Features\n(max_depth={best_params['max_depth']}, min_samples_leaf={best_params['min_samples_leaf']})")
plt.tight_layout()

# --- SAVE TREE IMAGE ---
tree_plot_path = os.path.join(SAVE_DIR, "tree_umap_visualization.png")
plt.savefig(tree_plot_path, dpi=300)
plt.show()

print(f"✅ Tree and UMAP model saved to: {SAVE_DIR}")
print(f"🖼️  Tree visualization saved as: {tree_plot_path}")


In [ ]:
def generate_rule_function(tree, feature_names=["UMAP0", "UMAP1"]):
    tree_ = tree.tree_
    feature_name = [feature_names[i] if i != _tree.TREE_UNDEFINED else "undefined!" for i in tree_.feature]
    def recurse(node, depth):
        indent = "    " * depth
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name = feature_name[node]
            threshold = tree_.threshold[node]
            return (
                f"{indent}if x[{feature_names.index(name)}] <= {threshold:.6f}:\n" +
                recurse(tree_.children_left[node], depth + 1) +
                f"{indent}else:\n" +
                recurse(tree_.children_right[node], depth + 1)
            )
        else:
            pred_class = np.argmax(tree_.value[node][0])
            return f"{indent}return {pred_class}  # MES {pred_class}\n"
    function_code = "def rule_based_predict_best(x):\n" + recurse(0, 1)
    return function_code

rule_function_code = generate_rule_function(tree_umap)
print(rule_function_code)
exec(rule_function_code, globals())

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import os

# --- Set Save Path ---
SAVE_DIR = f"./Result/CrossDataset/{experiment_saved_name}/"
os.makedirs(SAVE_DIR, exist_ok=True)
REPORT_PATH = os.path.join(SAVE_DIR, "evaluation_report.txt")

# --- Final Predictions ---
threshold = best_params['threshold']
y_pred_rule_umap = np.array([rule_based_predict_best(row) for row in X_test_umap])
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)
y_pred_combined = np.where(np.max(y_pred_proba, axis=1) < threshold, y_pred_rule_umap, y_pred_hybrid)
y_pred_override = np.array([hybrid_class if rule_class != hybrid_class else rule_class
                            for rule_class, hybrid_class in zip(y_pred_rule_umap, y_pred_hybrid)])
y_true = np.argmax(y_test_cat, axis=1)

# --- Generate Reports ---
report_hybrid = classification_report(y_true, y_pred_hybrid, digits=4)
report_combined = classification_report(y_true, y_pred_combined, digits=4)
report_override = classification_report(y_true, y_pred_override, digits=4)
report_rule = classification_report(y_true, y_pred_rule_umap, digits=4)

# --- Print to Console ---
print("📊 Hybrid Only:\n", report_hybrid)
print("📊 Rule-Aware Hybrid (Confidence-Guided Fallback):\n", report_combined)
print("📊 Rule-Aware Hybrid (Override When Agree):\n", report_override)
print("📊 Rule Only:\n", report_rule)

# --- Save to TXT File ---
with open(REPORT_PATH, 'w') as f:
    f.write("📊 Hybrid Only:\n")
    f.write(report_hybrid + "\n\n")
    f.write("📊 Rule-Aware Hybrid (Confidence-Guided Fallback):\n")
    f.write(report_combined + "\n\n")
    f.write("📊 Rule-Aware Hybrid (Override When Agree):\n")
    f.write(report_override + "\n\n")
    f.write("📊 Rule Only:\n")
    f.write(report_rule + "\n")

print(f"\n✅ Evaluation report saved to: {REPORT_PATH}")


In [ ]:
from flask import Flask, request, jsonify
import openai
import threading

openai.api_key = "YOUR_OPENAI_API_KEY_HERE"  # or from environment

app = Flask(__name__)

@app.route("/predict", methods=["POST"])
def predict():
    prompt = request.json.get("prompt", "")
    try:
        res = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=10
        )
        content = res['choices'][0]['message']['content']
        # Try to find valid class (0–3)
        for token in content.split():
            if token.isdigit() and int(token) in [0, 1, 2, 3]:
                return jsonify({"class": int(token), "raw": content})
        return jsonify({"class": None, "raw": content})
    except Exception as e:
        return jsonify({"error": str(e)})

# ✅ Run in separate thread
def run_flask():
    app.run(host='0.0.0.0', port=5000)

threading.Thread(target=run_flask).start()



In [ ]:
import os
import json
import numpy as np
from sklearn.metrics import classification_report

# --- Ensure UMAP projection is applied to test handcrafted features ---
X_feat_test_umap = umap_reducer.transform(X_feat_test_scaled)
y_true = np.argmax(y_test_cat, axis=1)  # <-- FIXED

# --- Predict using hybrid model ---
y_pred_proba = model_hybrid.predict(
    [X_img_test, X_feat_test_scaled, X_feat_test_umap], verbose=0
)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)
confidences = np.max(y_pred_proba, axis=1)

# --- Rule-based prediction ---
y_pred_rule_umap = np.array([rule_based_predict_best(row) for row in X_feat_test_umap])

# --- Fusion logic ---
threshold = best_params.get("threshold", 0.55)
y_pred_combined = np.where(confidences < threshold, y_pred_rule_umap, y_pred_hybrid)
y_pred_agree = np.array([
    rule if rule == model else rule
    for rule, model in zip(y_pred_rule_umap, y_pred_hybrid)
])

# --- Print evaluation ---
print("Hybrid Only:\n", classification_report(y_true, y_pred_hybrid, digits=4))
print("Rule-Aware Hybrid (Confidence-Guided):\n", classification_report(y_true, y_pred_combined, digits=4))
print("Rule-Aware Hybrid (Agreement Override):\n", classification_report(y_true, y_pred_agree, digits=4))
print("Rule Only:\n", classification_report(y_true, y_pred_rule_umap, digits=4))

# --- Save evaluation report to file ---
output_dir = f"./Result/CrossDataset/{experiment_saved_name}/"
os.makedirs(output_dir, exist_ok=True)

report_file = os.path.join(output_dir, "classification_report.txt")
with open(report_file, "w") as f:
    f.write("Hybrid Only:\n")
    f.write(classification_report(y_true, y_pred_hybrid, digits=4))
    f.write("\nRule-Aware Hybrid (Confidence-Guided):\n")
    f.write(classification_report(y_true, y_pred_combined, digits=4))
    f.write("\nRule-Aware Hybrid (Agreement Override):\n")
    f.write(classification_report(y_true, y_pred_agree, digits=4))
    f.write("\nRule Only:\n")
    f.write(classification_report(y_true, y_pred_rule_umap, digits=4))

print(f"✅ Classification report saved to: {report_file}")

# --- Save JSONL log for LLM feedback learning ---
log_file_path = os.path.join(output_dir, "llm_feedback_log_mixed.jsonl")
with open(log_file_path, "w") as f:
    for i in range(len(y_true)):
        entry = {
            "model_prediction": int(y_pred_hybrid[i]),
            "rule_prediction": int(y_pred_rule_umap[i]),
            "final_fusion_prediction": int(y_pred_combined[i]),
            "true_label": int(y_true[i]),
            "confidence": float(confidences[i]),
            "umap_0": float(X_feat_test_umap[i][0]),
            "umap_1": float(X_feat_test_umap[i][1]),
            "features": X_feat_test_scaled[i].tolist(),
            "feedback": ""
        }

        # Label feedback
        if y_pred_combined[i] != y_true[i]:
            if confidences[i] < threshold and y_pred_rule_umap[i] == y_true[i]:
                entry["feedback"] = "Should have used the rule-based prediction. Confidence was low."
            elif confidences[i] >= threshold and y_pred_hybrid[i] == y_true[i]:
                entry["feedback"] = "Correctly used the model prediction. Confidence was high."
            else:
                entry["feedback"] = "Incorrect prediction. Consider learning from UMAP and features."
        else:
            entry["feedback"] = "Correct prediction."

        f.write(json.dumps(entry) + "\n")

print(f"✅ Logged {len(y_true)} entries to {log_file_path}")


In [ ]:
import openai
import json
import time
from tqdm import tqdm

# ✅ Set your OpenAI API Key
openai.api_key = "YOUR_OPENAI_API_KEY_HERE"  # Replace with your OpenAI key

# ✅ Path to your saved .jsonl log
log_file_path = f"./Result/CrossDataset/{experiment_saved_name}/llm_feedback_log_mixed.jsonl"

# ✅ Load prediction entries
with open(log_file_path, "r") as f:
    entries = [json.loads(line) for line in f]

# ✅ Function to build prompt from entry
def build_prompt(entry):
    return f"""You are a colonoscopy diagnosis assistant trained to override model predictions when necessary.

Model prediction: {entry['model_prediction']}
Rule prediction: {entry['rule_prediction']}
Confidence: {entry['confidence']:.2f}
UMAP: ({entry['umap_0']:.2f}, {entry['umap_1']:.2f})
Handcrafted features: {entry['features']}

True label: {entry['true_label']}

Task: Given all the information, what should the final predicted MES class be?
Only respond with a single number: 0, 1, 2, or 3.
"""

# ✅ Function to get LLM recommendation
def query_gpt(prompt, model="gpt-3.5-turbo", retries=3):
    for attempt in range(retries):
        try:
            response = openai.ChatCompletion.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            return response.choices[0].message["content"].strip()
        except Exception as e:
            print(f"Error: {e}. Retrying...")
            time.sleep(1)
    return None

# ✅ Run through all entries and get LLM-based decisions
llm_preds = []
for entry in tqdm(entries, desc="LLM predicting"):
    prompt = build_prompt(entry)
    prediction = query_gpt(prompt)
    try:
        pred_int = int(prediction)
        if pred_int in [0, 1, 2, 3]:
            llm_preds.append(pred_int)
        else:
            llm_preds.append(entry["final_fusion_prediction"])  # fallback
    except:
        llm_preds.append(entry["final_fusion_prediction"])  # fallback

# ✅ Calculate accuracy of LLM override agent
true_labels = [e["true_label"] for e in entries]
correct = sum([int(p == t) for p, t in zip(llm_preds, true_labels)])
accuracy = correct / len(true_labels)

print(f"✅ LLM Agent Accuracy: {accuracy:.4f}")


In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from sklearn.metrics import accuracy_score, classification_report

# --- Setup ---
output_dir = f"./Result/CrossDataset/{experiment_saved_name}/"
os.makedirs(output_dir, exist_ok=True)

# ✅ Load previous log
log_file_path = os.path.join(output_dir, "llm_feedback_log_mixed.jsonl")
with open(log_file_path, "r") as f:
    entries = [json.loads(line) for line in f]

# ✅ Update and overwrite JSONL with LLM predictions
updated_log_path = os.path.join(output_dir, "llm_feedback_log_updated_mixed.jsonl")
with open(updated_log_path, "w") as f:
    for entry, llm_pred in zip(entries, llm_preds):
        entry["llm_prediction"] = llm_pred
        entry["llm_correct"] = int(llm_pred == entry["true_label"])
        f.write(json.dumps(entry) + "\n")

print(f"✅ Updated JSONL written to: {updated_log_path}")

# ✅ Collect predictions and labels
fusion_preds = [e["final_fusion_prediction"] for e in entries]
rule_preds = [e["rule_prediction"] for e in entries]
model_preds = [e["model_prediction"] for e in entries]
llm_preds = [e.get("llm_prediction", e["final_fusion_prediction"]) for e in entries]
true_labels = [e["true_label"] for e in entries]

# ✅ Per-class accuracy
methods = {
    "Model": model_preds,
    "Rule": rule_preds,
    "Fusion": fusion_preds,
    "LLM": llm_preds
}
class_labels = [0, 1, 2, 3]
acc_by_class = defaultdict(dict)

for method, preds in methods.items():
    for cls in class_labels:
        cls_indices = [i for i, t in enumerate(true_labels) if t == cls]
        cls_correct = sum([preds[i] == true_labels[i] for i in cls_indices])
        acc_by_class[method][cls] = cls_correct / len(cls_indices) if cls_indices else 0

# ✅ Bar Plot per-class accuracy
x = np.arange(len(class_labels))
width = 0.2
plt.figure(figsize=(10, 6))
for i, (method, accs) in enumerate(acc_by_class.items()):
    plt.bar(x + i*width, [accs[cls] for cls in class_labels], width=width, label=method)

plt.xticks(x + 1.5*width, [f"MES {cls}" for cls in class_labels])
plt.ylabel("Accuracy")
plt.title("Accuracy by MES Class per Method")
plt.legend()
plt.grid(True, axis='y')
plt.tight_layout()

# ✅ Save plot
plot_path = os.path.join(output_dir, "accuracy_by_class_per_method.png")
plt.savefig(plot_path)
plt.show()
print(f"✅ Saved accuracy plot to: {plot_path}")

# ✅ Print and save classification reports
def print_and_save_report(name, preds):
    acc = accuracy_score(true_labels, preds)
    report = classification_report(true_labels, preds, target_names=[f"MES {i}" for i in range(4)], digits=4)
    print(f"\n🔍 {name} Accuracy: {acc:.4f}")
    print(report)

    report_path = os.path.join(output_dir, f"{name.lower().replace(' ', '_')}_report.txt")
    with open(report_path, "w") as f:
        f.write(f"{name} Accuracy: {acc:.4f}\n\n")
        f.write(report)
    print(f"✅ Saved report to: {report_path}")

print_and_save_report("Model Only", model_preds)
print_and_save_report("Rule-Based", rule_preds)
print_and_save_report("Fusion", fusion_preds)
print_and_save_report("LLM Override", llm_preds)


In [ ]:
# === STEP 1: Model Prediction on Training Set ===
# y_pred_proba_train: softmax probabilities
import json
import os

def create_feedback_jsonl(
    filename,
    y_true,
    y_model_pred,
    y_rule_pred,
    proba,
    umap_feat,
    handcrafted_feat,
    save_path=f"./Result/CrossDataset/{experiment_saved_name}/"
):
    assert len(y_true) == len(y_model_pred) == len(y_rule_pred) == len(proba) == len(umap_feat) == len(handcrafted_feat)
    records = []
    for i in range(len(y_true)):
        record = {
            "true_label": int(y_true[i]),
            "model_pred": int(y_model_pred[i]),
            "rule_pred": int(y_rule_pred[i]),
            "confidence": float(np.max(proba[i])),
            "proba": list(map(float, proba[i])),
            "umap_0": float(umap_feat[i][0]),
            "umap_1": float(umap_feat[i][1]),
            "features": list(map(float, handcrafted_feat[i]))
        }
        records.append(record)
    
    full_path = os.path.join(save_path, filename)
    with open(full_path, "w") as f:
        for rec in records:
            f.write(json.dumps(rec) + "\n")
    
    print(f"✅ Saved {len(records)} samples to {full_path}")

# --- Predict on training set using all 3 branches ---
y_pred_proba_train = model_hybrid.predict(
    [X_img_train_bal, X_feat_train_bal, X_train_umap],
    verbose=1
)
y_pred_hybrid_train = np.argmax(y_pred_proba_train, axis=1)

# === STEP 2: Rule-Based Prediction on UMAP Train ===
y_pred_rule_umap_train = np.array([rule_based_predict_best(row) for row in X_train_umap])

# === STEP 3: Save feedback_train.jsonl ===
create_feedback_jsonl(
    filename="feedback_train.jsonl",
    y_true=y_train_bal,                       # SMOTE-balanced true labels
    y_model_pred=y_pred_hybrid_train,         # model prediction
    y_rule_pred=y_pred_rule_umap_train,       # rule-based prediction
    proba=y_pred_proba_train,                 # model probabilities
    umap_feat=X_train_umap,                   # 2D UMAP features
    handcrafted_feat=X_feat_train_bal      # original scaled 20D handcrafted features
)

In [ ]:
import os
import json
import numpy as np

# Define base path for saving
base_dir = f"./Result/CrossDataset/{experiment_saved_name}/"

def create_feedback_jsonl(filename, y_true, y_model_pred, y_rule_pred, proba, umap_feat, handcrafted_feat):
    records = []
    for i in range(len(y_true)):
        entry = {
            "true_label": int(y_true[i]),
            "model_prediction": int(y_model_pred[i]),
            "rule_prediction": int(y_rule_pred[i]),
            "confidence": float(np.max(proba[i])),
            "umap_0": float(umap_feat[i, 0]),
            "umap_1": float(umap_feat[i, 1]),
            "features": [float(f) for f in handcrafted_feat[i]]
        }
        records.append(entry)

    path = os.path.join(base_dir, filename)
    with open(path, "w") as f:
        for row in records:
            f.write(json.dumps(row) + "\n")
    print(f"✅ Saved {len(records)} samples to {path}")

# === GENERATE TESTING SET FEEDBACK FILE ===
create_feedback_jsonl(
    filename="feedback_test.jsonl",
    y_true=y_test_encoded,
    y_model_pred=y_pred_hybrid,
    y_rule_pred=y_pred_rule_umap,
    proba=y_pred_proba,
    umap_feat=X_feat_test_umap,
    handcrafted_feat=X_feat_test_scaled
)


In [ ]:
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import os
import joblib
import warnings
from hashlib import sha1

warnings.filterwarnings("ignore")

# === PATHS ===
train_file = f"./Result/CrossDataset/{experiment_saved_name}/feedback_train.jsonl"
test_file  = f"./Result/CrossDataset/{experiment_saved_name}/feedback_test.jsonl"
save_dir   = f"./Result/CrossDataset/{experiment_saved_name}/"
os.makedirs(save_dir, exist_ok=True)

# === LOAD FUNCTION ===
def load_feedback_jsonl(path):
    with open(path, "r") as f:
        data = [json.loads(line.strip()) for line in f]
    df = pd.DataFrame(data)
    df["label"] = df["true_label"]
    return df

df_train = load_feedback_jsonl(train_file)
df_test  = load_feedback_jsonl(test_file)
df_test_orig = df_test.copy()  # Freeze

# === FEATURE ENCODING ===
def encode_features(df):
    df_feat = df[["confidence", "umap_0", "umap_1"]].copy()
    for i in range(20):
        df_feat[f"f{i}"] = df["features"].apply(lambda x: x[i])
    return df_feat.values, df["label"].values

X_train, y_train = encode_features(df_train)
X_test, y_test   = encode_features(df_test_orig)

# === SCALING ===
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# === TRAIN LOOP ===
loop = 0
acc_list = []
known_misclassified = set()

while True:
    clf = lgb.LGBMClassifier(random_state=42, class_weight='balanced')

    clf.fit(X_train_scaled, y_train)

    y_pred = clf.predict(X_test_scaled)
    y_proba = clf.predict_proba(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    acc_list.append(acc)

    print(f"🔁 Loop {loop+1}: Accuracy = {acc:.4f}")
    if acc >= 0.90:
        print("✅ Target reached.")
        break

    misclassified = df_test_orig[y_pred != y_test].copy()
    misclassified["hash"] = misclassified.apply(
        lambda row: sha1(json.dumps(row.to_dict(), sort_keys=True).encode()).hexdigest(), axis=1
    )
    new_errors = misclassified[~misclassified["hash"].isin(known_misclassified)]

    if new_errors.empty:
        print("⚠️ No new unique misclassified samples to learn from.")
        break

    print(f"➕ Adding {len(new_errors)} new feedback samples")
    known_misclassified.update(new_errors["hash"])
    df_train = pd.concat([df_train, new_errors.drop(columns=["hash"])], ignore_index=True)

    X_train, y_train = encode_features(df_train)
    X_train_scaled = scaler.fit_transform(X_train)

    loop += 1

# === SAVE FINAL AGENT ===
clf.booster_.save_model(os.path.join(save_dir, "feedback_agent.txt"))
joblib.dump(scaler, os.path.join(save_dir, "scaler_agent.pkl"))

# === SAVE LEARNING CURVE ===
plt.figure(figsize=(10, 5))
plt.plot(acc_list, marker='o', label="Test Accuracy")
plt.axhline(0.90, color='red', linestyle='--', label="Target")
plt.title("Agent Continual Learning Curve")
plt.xlabel("Iteration")
plt.ylabel("Accuracy")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "agent_learning_curve.png"))

# === FINAL REPORT ===
report = classification_report(y_test, y_pred, digits=4, output_dict=True)
report_df = pd.DataFrame(report).T
report_df.to_csv(os.path.join(save_dir, "agent_final_classification_report.csv"))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import joblib
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import KFold
import lightgbm as lgb

# === PATHS ===
save_dir = f"./Result/CrossDataset/{experiment_saved_name}/"
test_file = os.path.join(save_dir, "feedback_test.jsonl")
agent_file = os.path.join(save_dir, "feedback_agent.txt")
scaler_file = os.path.join(save_dir, "scaler_agent.pkl")

# === LOAD DATASET ===
with open(test_file, "r") as f:
    test_data = [json.loads(line.strip()) for line in f]
df_all = pd.DataFrame(test_data)
df_all["label"] = df_all["true_label"]

# === FEATURE ENCODER ===
def encode_features(df):
    df_feat = df[["confidence", "umap_0", "umap_1"]].copy()
    for i in range(20):
        df_feat[f"f{i}"] = df["features"].apply(lambda x: x[i])
    return df_feat.values, df["label"].values

X, y = encode_features(df_all)
scaler = joblib.load(scaler_file)
X_scaled = scaler.transform(X)

# === LOAD AGENT ===
agent = lgb.Booster(model_file=agent_file)

# === K-FOLD EVALUATION ===
n_classes = 4
class_names = [f"MES {i}" for i in range(n_classes)]
kf = KFold(n_splits=10, shuffle=True, random_state=42)

fold = 1
for train_idx, test_idx in kf.split(X_scaled, y):
    print(f"\n===== Fold {fold} =====")

    X_test, y_test = X_scaled[test_idx], y[test_idx]

    # Predict with existing agent
    y_proba = agent.predict(X_test)
    y_pred = np.argmax(y_proba, axis=1)
    y_true_oh = label_binarize(y_test, classes=list(range(n_classes)))

    # === Per-Class Custom Metrics ===
    cm = confusion_matrix(y_test, y_pred)
    metrics_dict = {}
    for i in range(n_classes):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = np.sum(cm) - (TP + FP + FN)
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        npv       = TN / (TN + FN) if (TN + FN) > 0 else 0
        ppv       = precision
        accuracy  = (TP + TN) / (TP + TN + FP + FN)
        metrics_dict[f"MES {i}"] = {
            "precision": precision,
            "recall": recall,
            "f1-score": f1,
            "npv": npv,
            "ppv": ppv,
            "accuracy": accuracy
        }

    # Macro Average
    all_vals = pd.DataFrame(metrics_dict).T
    metrics_dict["Overall"] = {
        metric: all_vals[metric].mean() for metric in all_vals.columns
    }
    summary = pd.DataFrame(metrics_dict).T

    # Save CSV per fold
    summary.to_csv(os.path.join(save_dir, f"agent_eval_summary_fold{fold}.csv"))
    print("📊 Evaluation Summary (Fold", fold, "):\n", summary.round(4))

    summary.to_csv(os.path.join(save_dir, f"agent_eval_summary_fold{fold}.csv"))
    print("📊 Evaluation Summary (Fold", fold, "):\n", summary.round(4))
    
    # === Dump to TXT cumulative file ===
    txt_path = os.path.join(save_dir, "agent_eval_summary_all.txt")
    with open(txt_path, "a") as f_out:
        f_out.write(f"\n===== Fold {fold} =====\n")
        f_out.write(summary.round(4).to_string())
        f_out.write("\n")
    
    # === SAVE  ===
    if fold == 10:
        # Radar Chart
        metrics = ["precision", "recall", "f1-score", "npv"]
        angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
        angles += angles[:1]
        plt.figure(figsize=(8, 6))
        ax = plt.subplot(111, polar=True)
        for i in range(n_classes):
            label = f"MES {i}"
            values = summary.loc[label, metrics].values.tolist()
            values += values[:1]
            ax.plot(angles, values, label=label)
            ax.fill(angles, values, alpha=0.1)
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)
        plt.xticks(angles[:-1], metrics)
        plt.yticks([0.25, 0.5, 0.75, 1.0], ["0.25", "0.5", "0.75", "1.0"])
        plt.title("Radar Chart: Per-Class Evaluation (Fold 10)")
        plt.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, "agent_radar_chart_final.png"))
        plt.show()

        # Confusion Matrix
        fig, ax = plt.subplots(1, 2, figsize=(12, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax[0],
                    xticklabels=class_names, yticklabels=class_names)
        ax[0].set_title("Confusion Matrix (Fold 10)")
        ax[0].set_xlabel("Predicted")
        ax[0].set_ylabel("True")
        cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
        sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", ax=ax[1],
                    xticklabels=class_names, yticklabels=class_names)
        ax[1].set_title("Normalized Confusion Matrix (Fold 10)")
        ax[1].set_xlabel("Predicted")
        ax[1].set_ylabel("True")
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, "agent_confusion_matrix_final.png"))
        plt.show()

        # ROC Curve
        fpr = dict(); tpr = dict(); roc_auc = dict()
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_true_oh[:, i], y_proba[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])
        fpr["micro"], tpr["micro"], _ = roc_curve(y_true_oh.ravel(), y_proba.ravel())
        roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
        roc_auc["macro"] = roc_auc_score(y_true_oh, y_proba, average="macro")

        plt.figure(figsize=(8, 6))
        for i in range(n_classes):
            plt.plot(fpr[i], tpr[i], label=f"MES {i} (AUC = {roc_auc[i]:.2f})")
        plt.plot(fpr["micro"], tpr["micro"], color="black", linestyle="--", 
                 label=f"Micro Avg (AUC = {roc_auc['micro']:.2f})")
        plt.plot([0, 1], [0, 1], "k--", lw=1)
        plt.title("ROC Curves (Fold 10)")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.legend(loc="lower right")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, "agent_roc_curve_final.png"))
        plt.show()

    fold += 1


In [ ]:
# === Radar Chart ===
metrics = ["accuracy", "precision", "recall", "f1-score"]
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]  # to close the radar chart loop

plt.figure(figsize=(8, 6))
ax = plt.subplot(111, polar=True)

for i in range(n_classes):
    label = f"MES {i}"
    values = summary.loc[label, metrics].values.tolist()
    values += values[:1]
    ax.plot(angles, values, label=label)
    ax.fill(angles, values, alpha=0.1)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
plt.xticks(angles[:-1], metrics)
plt.yticks([0.25, 0.5, 0.75, 1.0], ["0.25", "0.5", "0.75", "1.0"])
plt.title("Radar Chart: Accuracy, Precision, Recall, F1-Score (Per Class)")
plt.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "agent_radar_chart_accuracy.png"))
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# === Define save directory ===
save_dir = f"./Result/CrossDataset/{experiment_saved_name}/"

# === Handcrafted feature names: 20D (from Wavelet + GLCM) ===
handcrafted_feature_names = [
    "LL_Mean", "LL_Std", "LL_Var", "LL_Entropy",
    "LH_Mean", "LH_Std", "LH_Var", "LH_Entropy",
    "HL_Mean", "HL_Std", "HL_Var", "HL_Entropy",
    "HH_Mean", "HH_Std", "HH_Var", "HH_Entropy",
    "HH_Energy",
    "GLCM_Contrast", "GLCM_Dissimilarity", "GLCM_Homogeneity"
]

# === Get feature importances using LightGBM Booster method ===
importances = agent.feature_importance()

# === Handcrafted features are from index 3 to 22 (inclusive) ===
handcrafted_importances = importances[3:23]

# === Sort descending ===
sorted_idx = np.argsort(handcrafted_importances)[::-1]
sorted_names = [handcrafted_feature_names[i] for i in sorted_idx]
sorted_vals = handcrafted_importances[sorted_idx]

# === Plot bar chart ===
plt.figure(figsize=(10, 6))
plt.barh(sorted_names, sorted_vals, color='skyblue')
plt.gca().invert_yaxis()
plt.xlabel("Importance Score")
plt.title("🔬 Handcrafted Feature Importance (LightGBM Agent)")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "agent_handcrafted_feature_importance.png"))
plt.show()


In [ ]:
print("Feedback log sample size:", len(current_data))  # From feedback loop
print("Validation set used during agent training:", len(y_val))  # Typically ~20% of that
print("Real test set used for evaluation:", len(y_true))  # 145


In [ ]:
import cv2
import numpy as np
import joblib
import os
import pandas as pd
import pywt
import sys
import scipy.stats
import warnings

# --- Suppress Warnings ---
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

try:
    import tensorflow as tf
    tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
    from skimage.feature import graycomatrix, graycoprops
    import lightgbm as lgb
except ImportError as e:
    print(f"❌ ERROR: Library penting tidak ditemukan. {e}")
    print("Harap instal: pip install tensorflow scikit-image pywavelets lightgbm")
    sys.exit(1)

# === KONFIGURASI & KONSTANTA ===
IMG_SIZE = (224, 224)
WAVELET = 'db1'
experiment_saved_name = "TrainingMixedTesting1" # Sesuaikan jika perlu

# === PATH KE SEMUA 5 MODEL ===
# Ambil base path dari environment variable, atau gunakan default
model_dir = os.getenv("MODEL_DIR", f"./Result/CrossDataset/{experiment_saved_name}/")

# Model Langkah 1 (Hybrid Keras)
hybrid_model_path = os.path.join(model_dir, "TryFindingBestModel.h5")
umap_model_path = os.path.join(model_dir, "umap_model_mixed.pkl")
scaler_20_path = os.path.join(model_dir, "scaler_handcrafted_20.pkl") 

# Model Langkah 2 (LGBM Agent)
lgbm_agent_path = os.path.join(model_dir, "feedback_agent.txt")
scaler_23_path = os.path.join(model_dir, "scaler_agent.pkl")


# === MEMUAT SEMUA 5 MODEL ===
print("--- Memuat Semua 5 Komponen Model ---")
try:
    # Model Langkah 1
    keras_hybrid = tf.keras.models.load_model(hybrid_model_path, compile=False)
    umap_model = joblib.load(umap_model_path)
    scaler_20 = joblib.load(scaler_20_path)
    print("✅ Model Keras, UMAP, dan Scaler_20 (Langkah 1) dimuat.")
    
    # Model Langkah 2
    lgbm_agent = lgb.Booster(model_file=lgbm_agent_path)
    scaler_23 = joblib.load(scaler_23_path)
    print("✅ Model LGBM Agent dan Scaler_23 (Langkah 2) dimuat.")

    MODELS = {
        "keras_hybrid": keras_hybrid,
        "umap": umap_model,
        "scaler_20": scaler_20,
        "lgbm_agent": lgbm_agent,
        "scaler_23": scaler_23
    }
except Exception as e:
    print(f"\n❌ ERROR: Gagal memuat salah satu dari 5 model di direktori: {model_dir}")
    print(f"Detail Error: {e}")
    print("Pastikan semua file ini ada:")
    print(f"1. {os.path.basename(hybrid_model_path)}")
    print(f"2. {os.path.basename(umap_model_path)}")
    print(f"3. {os.path.basename(scaler_20_path)}")
    print(f"4. {os.path.basename(lgbm_agent_path)}")
    print(f"5. {os.path.basename(scaler_23_path)}")
    sys.exit(1)


# === FUNGSI EKSTRAKSI FITUR (dari Arsitektur ke-2) ===
def extract_wavelet_stats(image):
    if len(image.shape) == 3 and image.shape[2] == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image
    coeffs2 = pywt.dwt2(gray, WAVELET)
    LL, (LH, HL, HH) = coeffs2
    def stats(subband):
        if subband.size == 0: return [0.0, 0.0, 0.0, 0.0]
        flat_band = np.abs(subband.flatten()) + 1e-9
        return [np.mean(subband), np.std(subband), np.var(subband), scipy.stats.entropy(flat_band)]
    features = []
    for band in [LL, LH, HL, HH]:
        features.extend(stats(band))
    hh_energy = np.sum(np.square(HH)) if HH.size > 0 else 0.0
    features.append(hh_energy)
    return features # Total 17 fitur

def extract_glcm_features_extended(image):
    if len(image.shape) == 3 and image.shape[2] == 3:
        gray_8bit = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray_8bit = image
    if gray_8bit.dtype != np.uint8:
        if gray_8bit.max() <= 1.0 and gray_8bit.min() >= 0.0:
            gray_8bit = (gray_8bit * 255).astype(np.uint8)
        else:
            gray_8bit = gray_8bit.astype(np.uint8)
    if gray_8bit.size == 0 or gray_8bit.shape[0] < 5 or gray_8bit.shape[1] < 5:
        return [0.0, 0.0, 0.0]
    angles = [0, np.pi/4, np.pi/2]
    try:
        glcm = graycomatrix(gray_8bit, distances=[5], angles=angles, levels=256, symmetric=True, normed=True)
        contrast = graycoprops(glcm, 'contrast').mean()
        dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
        homogeneity = graycoprops(glcm, 'homogeneity').mean()
        return [contrast, dissimilarity, homogeneity] # Total 3 fitur
    except ValueError as e:
        return [0.0, 0.0, 0.0]
        
def extract_20_handcrafted_features(image):
    """Menggabungkan wavelet dan glcm untuk mendapatkan 20 fitur."""
    wavelet_feats = extract_wavelet_stats(image)     # 17 fitur
    glcm_feats = extract_glcm_features_extended(image) # 3 fitur
    return np.array(wavelet_feats + glcm_feats)    # Total 20 fitur


# === FUNGSI PIPELINE 2-LANGKAH ===

def run_step_1_keras(image_path):
    """
    Menjalankan pipeline Keras (Langkah 1) untuk mendapatkan
    prediksi awal dan fitur-fitur yang diperlukan untuk Langkah 2.
    """
    try:
        # 1. Pra-pemrosesan Gambar
        img = cv2.imread(image_path)
        if img is None:
            print(f"Error: Gagal membaca gambar {image_path}")
            return None
            
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # TODO: Verifikasi dimensi cropping ini (30:430, 200:550)
        # Jika gambar input Anda sudah di-crop, Anda mungkin bisa melewati ini
        try:
            cropped = img_rgb[30:430, 200:550]
        except IndexError:
            print("Warning: Gagal cropping, menggunakan gambar asli.")
            cropped = img_rgb
            
        resized = cv2.resize(cropped, IMG_SIZE)
        
        # Input 1 (Gambar)
        input_1_image = np.expand_dims(resized.astype(np.float32) / 255.0, axis=0)
        
        # 2. Ekstraksi Fitur (untuk Input 2)
        features_20d_raw = extract_20_handcrafted_features(resized).reshape(1, -1)
        
        # Scaling Fitur 20
        input_2_features_scaled = MODELS["scaler_20"].transform(features_20d_raw)

        # 3. Transformasi UMAP (untuk Input 3)
        input_3_umap = MODELS["umap"].transform(input_2_features_scaled)
        
        # 4. Prediksi Keras (Langkah 1)
        print("... Menjalankan prediksi Keras (Langkah 1)...")
        all_keras_inputs = [input_1_image, input_2_features_scaled, input_3_umap]
        keras_proba = MODELS["keras_hybrid"].predict(all_keras_inputs, verbose=0)[0]
        
        # 5. Kumpulkan semua output untuk Langkah 2
        outputs = {
            "keras_proba": keras_proba,
            "scaled_20_features": input_2_features_scaled,
            "umap_features": input_3_umap,
            "raw_20_features": features_20d_raw[0] # Untuk logging/hasil
        }
        return outputs

    except Exception as e:
        print(f"❌ Error saat menjalankan Langkah 1 (Keras): {e}")
        import traceback
        traceback.print_exc()
        return None

def run_step_2_lgbm(step_1_outputs):
    """
    Menjalankan pipeline LGBM (Langkah 2) menggunakan output dari Langkah 1.
    """
    try:
        # Ambil data dari Langkah 1
        keras_proba = step_1_outputs["keras_proba"]
        scaled_20_features = step_1_outputs["scaled_20_features"][0] # Ambil array 1D (20,)
        umap_features = step_1_outputs["umap_features"][0]       # Ambil array 1D (2,)

        # --- Buat 23 Fitur untuk Agen LGBM ---
        # 1. Confidence (dari Keras)
        confidence = np.max(keras_proba)
        # 2. UMAP 0
        umap_0 = umap_features[0]
        # 3. UMAP 1
        umap_1 = umap_features[1]
        
        # Gabungkan semua 23 fitur
        lgbm_input_raw_23 = np.concatenate(
            ([confidence, umap_0, umap_1], scaled_20_features)
        ).reshape(1, -1) # Bentuk (1, 23)
        
        # Scaling 23 Fitur (menggunakan scaler_agent.pkl)
        lgbm_input_scaled_23 = MODELS["scaler_23"].transform(lgbm_input_raw_23)
        
        # Prediksi LGBM (Langkah 2)
        print("... Menjalankan prediksi LGBM Agent (Langkah 2)...")
        final_proba = MODELS["lgbm_agent"].predict(lgbm_input_scaled_23)[0]
        return final_proba

    except Exception as e:
        print(f"❌ Error saat menjalankan Langkah 2 (LGBM): {e}")
        return None


# === FUNGSI PREDIKSI UTAMA (Menggantikan file pertama Anda) ===

def predict_from_image(image_path):
    """
    Fungsi prediksi 2-langkah lengkap (Keras -> LGBM)
    """
    
    print(f"\n--- Memulai Prediksi 2-Langkah untuk: {os.path.basename(image_path)} ---")
    
    # 1. Jalankan Langkah 1 (Keras)
    step_1_outputs = run_step_1_keras(image_path)
    if step_1_outputs is None:
        return {"error": "Gagal pada Langkah 1 (Keras)"}
        
    keras_proba = step_1_outputs["keras_proba"]
    keras_label = int(np.argmax(keras_proba))
    
    # 2. Jalankan Langkah 2 (LGBM)
    final_proba = run_step_2_lgbm(step_1_outputs)
    if final_proba is None:
        return {"error": "Gagal pada Langkah 2 (LGBM)"}
        
    final_label = int(np.argmax(final_proba))
    
    # 3. Siapkan Hasil (Mirip dengan file pertama Anda, tapi sekarang dengan data nyata)
    
    # Dapatkan nama fitur dari scaler_20
    # Ini sedikit rumit karena kita perlu mencocokkan nama
    handcrafted_feature_names = [
        "LL_Mean", "LL_Std", "LL_Var", "LL_Entropy",
        "LH_Mean", "LH_Std", "LH_Var", "LH_Entropy",
        "HL_Mean", "HL_Std", "HL_Var", "HL_Entropy",
        "HH_Mean", "HH_Std", "HH_Var", "HH_Entropy", "HH_Energy",
        "GLCM_Contrast", "GLCM_Dissimilarity", "GLCM_Homogeneity"
    ] # Total 20 fitur
    
    raw_features = step_1_outputs["raw_20_features"]
    all_features_dict = {handcrafted_feature_names[i]: float(raw_features[i]) for i in range(len(raw_features))}

    # Dapatkan feature importances dari LGBM
    # Fitur LGBM: [confidence, umap_0, umap_1, f0, f1, ..., f19]
    importances = MODELS["lgbm_agent"].feature_importance(importance_type='gain')
    
    # Ambil importances HANYA untuk 20 fitur handcrafted (indeks 3 s/d 22)
    handcrafted_importances = importances[3:] 
    
    # Urutkan berdasarkan importances
    sorted_idx = np.argsort(handcrafted_importances)[::-1][:5] # Ambil top 5
    top5_features = [
        (handcrafted_feature_names[i], all_features_dict[handcrafted_feature_names[i]]) 
        for i in sorted_idx
    ]

    nama_kelas = {0: 'MES0', 1: 'MES1', 2: 'MES2', 3: 'MES3'}

    results = {
        "prediction_label": nama_kelas.get(final_label, "Unknown"),
        "prediction_index": final_label,
        "prediction": [float(p) for p in final_proba],
        "top_5": top5_features,
        "internal_keras_prediction": {
            "label": nama_kelas.get(keras_label, "Unknown"),
            "index": keras_label,
            "scores": [float(p) for p in keras_proba]
        },
        "all_features": all_features_dict
    }
    
    return results


def main():

    IMAGE_PATH  = "./Result/CrossDataset/TrainingMixedTesting1/class_1.jpg"

    print("=" * 70)
    print("🖼️   COLONOSCOPY MES CLASSIFICATION - TESTING")
    print("=" * 70)
    
    # Validasi file gambar
    if not os.path.exists(IMAGE_PATH):
        print(f"\n❌ ERROR: File gambar tidak ditemukan di: {IMAGE_PATH}")
        print(f"Pastikan file gambar ada di lokasi yang benar.")
        return
    
    try:
        file_size_kb = os.path.getsize(IMAGE_PATH) / 1024
        print(f"\n📷 Gambar: {IMAGE_PATH}")
        print(f"📊 Ukuran file: {file_size_kb:.2f} KB")
    except Exception as e:
        print(f"\n❌ ERROR: Tidak dapat mengakses file gambar: {e}")
        return
    
    # Jalankan prediksi
    print("\n" + "=" * 70)
    print("🔄 Memulai prediksi (Ini mungkin perlu waktu untuk memuat model)...")
    print("=" * 70)
    
    results = predict_from_image(IMAGE_PATH)
    
    if results is None or "error" in results:
        print(f"\n❌ Prediksi gagal. Error: {results.get('error', 'Unknown')}")
        print("Periksa error message di atas untuk detail (misal: model tidak ditemukan).")
        return
    
    # Display results
    print("\n" + "=" * 70)
    print("✅ HASIL PREDIKSI")
    print("=" * 70)
    
    # Definisikan label
    mes_labels = {0: 'MES-0', 1: 'MES-1', 2: 'MES-2', 3: 'MES-3'}
    
    # === HASIL LANGKAH 1 (KERAS) ===
    # Mengakses nested dictionary 'internal_keras_prediction'
    keras_pred = results['internal_keras_prediction']['index']
    keras_proba_list = results['internal_keras_prediction']['scores']
    keras_proba_formatted = [f"{p:.4f}" for p in keras_proba_list] # Format proba

    print(f"\n🧠 LANGKAH 1 - Model Keras Hybrid:")
    print(f"   Prediksi: {keras_pred} ({mes_labels.get(keras_pred)})")
    print(f"   Probabilitas: {keras_proba_formatted}")
    
    # === HASIL LANGKAH 2 (LGBM) ===
    # Mengakses key utama
    final_pred = results['prediction_index']
    final_proba_list = results['prediction']
    final_proba_formatted = [f"{p:.4f}" for p in final_proba_list] # Format proba

    print(f"\n👨‍🏫 LANGKAH 2 - Agen LGBM (Refinement):")
    print(f"   Prediksi: {final_pred} ({mes_labels.get(final_pred)})")
    print(f"   Probabilitas: {final_proba_formatted}")
    
    # === KESIMPULAN ===
    print(f"\n📋 KESIMPULAN:")
    print(f"   Prediksi Akhir: {final_pred} ({mes_labels.get(final_pred, 'Unknown')})")
    print(f"   Confidence: {max(final_proba_list):.4f}")

    # === FITUR PENTING ===
    print(f"\n--- Fitur Handcrafted Paling Berpengaruh (Top 5) ---")
    try:
        for nama, nilai in results['top_5']:
            print(f"   - {nama:<20}: {nilai:.4f}")
    except Exception as e:
        print(f"   (Gagal menampilkan fitur: {e})")

    print("\n" + "=" * 70)

if __name__ == "__main__":
    main()



In [ ]:
# ===================================================================
# KODE GABUNGAN FINAL (TERMASUK PLOT GABUNGAN 1x2 dan 1x3)
# DENGAN UPDATE STYLE VISUALISASI (Font Besar & Tebal)
# ===================================================================

import os
import cv2
import numpy as np
import pywt
import scipy.stats
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from skimage.feature import graycomatrix, graycoprops
from tensorflow.keras.utils import to_categorical, custom_object_scope
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.tree import DecisionTreeClassifier, _tree, plot_tree
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
import itertools
import tqdm
import seaborn as sns
import pandas as pd
import joblib
import lightgbm as lgb
import tensorflow as tf
from sklearn.manifold import TSNE 
from tensorflow.keras.layers import Input, Dense, Concatenate, BatchNormalization, Dropout, Lambda
from transformers import TFViTModel, ViTFeatureExtractor
from itertools import cycle
import traceback 
import umap 
from sklearn.decomposition import PCA 

# --- KONFIGURASI ---
IMG_SIZE = (224, 224)
WAVELET = 'db1'
TRAIN_DIR = './MES Mixed Data' 
TEST_DIR  = './MES classification_20250313'
experiment_saved_name = "TrainingMixedTesting1"
SAVE_DIR = f"./Result/CrossDataset/{experiment_saved_name}/"
os.makedirs(SAVE_DIR, exist_ok=True) 

# ===================================================================
# BAGIAN 1: DEFINISI FUNGSI
# ===================================================================

def extract_wavelet_stats(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    coeffs2 = pywt.dwt2(gray, WAVELET)
    LL, (LH, HL, HH) = coeffs2
    def stats(subband):
        return [np.mean(subband), np.std(subband), np.var(subband), scipy.stats.entropy(np.abs(subband.flatten()) + 1e-6)]
    features = []
    for band in [LL, LH, HL, HH]: features.extend(stats(band))
    features.append(np.sum(np.square(HH)))
    return features

def extract_glcm_features_extended(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    angles = [0, np.pi/4, np.pi/2]
    gray_u8 = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    glcm = graycomatrix(gray_u8, distances=[5], angles=angles, levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').mean()
    dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    return [contrast, dissimilarity, homogeneity]

def load_dataset(folder_path):
    X_img, X_feat, y_label, img_paths = [], [], [], []
    if not os.path.isdir(folder_path): return None, None, None, None
    total_loaded = 0
    for label in sorted(os.listdir(folder_path)):
        label_path = os.path.join(folder_path, label)
        if not os.path.isdir(label_path): continue
        print(f"📂 Loading from folder: {label_path}")
        count_label = 0
        for fname in sorted(os.listdir(label_path)):
            img_path = os.path.join(label_path, fname)
            img = cv2.imread(img_path)
            if img is None: continue
            try:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                cropped = img[30:430, 200:550]
                resized = cv2.resize(cropped, IMG_SIZE)
            except: continue
            try:
                wavelet_feats = extract_wavelet_stats(resized)
                glcm_feats = extract_glcm_features_extended(resized)
                combined = wavelet_feats + glcm_feats
            except: continue
            X_img.append(resized)
            X_feat.append(combined)
            y_label.append(label)
            img_paths.append(img_path)
            count_label += 1
            total_loaded += 1
    if total_loaded == 0: return None, None, None, None
    return np.array(X_img), np.array(X_feat), np.array(y_label), np.array(img_paths)

# --- Layer Kustom ---
def GroupConv2D(filters, kernel_size, strides=(1, 1), padding='same', groups=3):
    def layer(x):
        return tf.keras.layers.Conv2D(filters, kernel_size, strides=strides, padding=padding)(x)
    return layer

def SE2MaxPooling2D(pool_size=(2, 2)):
    def layer(x): return tf.keras.layers.MaxPooling2D(pool_size=pool_size)(x)
    return layer

def SE2LiftingLayer(x):
    def layer(x): return x
    return layer

def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-8, 1.0)
        ce = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.math.pow(1 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=1))
    return loss

# ===================================================================
# BAGIAN 2: PIPELINE UTAMA
# ===================================================================

print("🚀 Memulai pipeline preprocessing...")
try:
    # --- TAHAP 1: PREPROCESSING ---
    X_img_train_raw, X_feat_train_raw, y_train_label, img_paths_train = load_dataset(TRAIN_DIR)
    X_img_test_raw,  X_feat_test_raw,  y_test_label,  img_paths_test  = load_dataset(TEST_DIR)

    if X_img_train_raw is None or X_img_test_raw is None: raise ValueError("Data loading failed.")

    X_img_train = X_img_train_raw.astype(np.float32) / 255.0
    X_img_test  = X_img_test_raw.astype(np.float32) / 255.0

    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train_label)
    y_test_encoded  = le.transform(y_test_label)
    class_names = le.classes_
    class_map = {i: class_names[i] for i in range(len(class_names))}

    scaler = StandardScaler()
    X_feat_train_scaled = scaler.fit_transform(X_feat_train_raw)
    X_feat_test_scaled  = scaler.transform(X_feat_test_raw)

    smote = SMOTE(random_state=42)
    X_feat_train_bal, y_train_bal = smote.fit_resample(X_feat_train_scaled, y_train_encoded)

    print("🌀 Menjalankan UMAP pada Handcrafted Features...")
    umap_reducer = umap.UMAP(n_neighbors=10, min_dist=0.05, n_components=2, metric='euclidean', random_state=42)
    X_train_umap = umap_reducer.fit_transform(X_feat_train_bal)
    X_test_umap_raw_features  = umap_reducer.transform(X_feat_test_scaled) 
    
    # Sinkronisasi
    min_len_test = min(len(X_test_umap_raw_features), len(y_test_encoded))
    X_test_umap_raw_features = X_test_umap_raw_features[:min_len_test]
    y_test_encoded = y_test_encoded[:min_len_test]
    X_img_test = X_img_test[:min_len_test]
    X_feat_test_scaled = X_feat_test_scaled[:min_len_test]
    X_img_test_raw = X_img_test_raw[:min_len_test]
    y_test_label = y_test_label[:min_len_test]
    
    # --- TAHAP 1B: UMAP PADA PIKSEL MENTAH ---
    print("🌀 Menjalankan PCA & UMAP pada Piksel Mentah...")
    n_samples = X_img_test.shape[0]
    X_img_test_flat = X_img_test.reshape(n_samples, -1)
    n_comp_pca = min(n_samples, 50) 
    pca = PCA(n_components=n_comp_pca, random_state=42)
    X_img_test_pca = pca.fit_transform(X_img_test_flat)
    
    umap_reducer_pixels = umap.UMAP(n_neighbors=10, min_dist=0.05, n_components=2, metric='euclidean', random_state=42)
    X_test_umap_raw_pixels = umap_reducer_pixels.fit_transform(X_img_test_pca)

    # --- TAHAP 3: LOAD MODEL & PREDICT ---
    KERAS_MODEL_PATH = os.path.join(SAVE_DIR, "TryFindingBestModel.h5")
    TREE_MODEL_PATH = os.path.join(SAVE_DIR, "rule_tree_mixed.pkl")
    AGENT_MODEL_PATH = os.path.join(SAVE_DIR, "feedback_agent.txt")
    AGENT_SCALER_PATH = os.path.join(SAVE_DIR, "scaler_agent.pkl")
    
    print("📦 Memuat model...")
    custom_objects = {'loss': focal_loss(2.5, 0.25), 'GroupConv2D': GroupConv2D, 'SE2MaxPooling2D': SE2MaxPooling2D, 'SE2LiftingLayer': SE2LiftingLayer}
    with custom_object_scope(custom_objects):
        model_hybrid = load_model(KERAS_MODEL_PATH)
    
    tree_umap = joblib.load(TREE_MODEL_PATH)
    agent_lgbm = lgb.Booster(model_file=AGENT_MODEL_PATH)
    scaler_agent = joblib.load(AGENT_SCALER_PATH)

    def rule_based_predict_best(x_row):
        return tree_umap.predict(x_row.reshape(1, -1))[0]

    y_pred_proba = model_hybrid.predict([X_img_test, X_feat_test_scaled, X_test_umap_raw_features], verbose=0)
    y_pred_hybrid = np.argmax(y_pred_proba, axis=1)
    y_pred_rule = np.array([rule_based_predict_best(row) for row in X_test_umap_raw_features])
    threshold = 0.55
    y_pred_hybrid_selector = np.where(np.max(y_pred_proba, axis=1) < threshold, y_pred_rule, y_pred_hybrid)

    confidences = np.max(y_pred_proba, axis=1)
    feat_cols = [f"f{i}" for i in range(X_feat_test_scaled.shape[1])]
    df_agent_input = pd.DataFrame(X_feat_test_scaled, columns=feat_cols)
    df_agent_input['confidence'] = confidences
    df_agent_input['umap_0'] = X_test_umap_raw_features[:, 0]
    df_agent_input['umap_1'] = X_test_umap_raw_features[:, 1]
    X_agent_input_scaled = scaler_agent.transform(df_agent_input[['confidence', 'umap_0', 'umap_1'] + feat_cols])
    
    y_proba_agent = np.array(agent_lgbm.predict(X_agent_input_scaled))
    y_pred_colonoscan = np.argmax(y_proba_agent, axis=1) if y_proba_agent.ndim > 1 else np.rint(y_proba_agent).astype(int)

    # --- TAHAP 4: EKSTRAKSI EMBEDDINGS ---
    print("✨ Mengekstrak Embeddings...")
    EMBEDDING_LAYER_NAME = 'dense_5' 
    embedding_model = Model(inputs=model_hybrid.inputs, outputs=model_hybrid.get_layer(EMBEDDING_LAYER_NAME).output)
    hybrid_embeddings = embedding_model.predict([X_img_test, X_feat_test_scaled, X_test_umap_raw_features], verbose=1)
    
    umap_reducer_embeddings = umap.UMAP(n_neighbors=10, min_dist=0.05, n_components=2, random_state=42)
    X_test_umap_hybrid_embeddings = umap_reducer_embeddings.fit_transform(hybrid_embeddings)

    # --- CETAK LAPORAN ---
    target_names = [class_map[i] for i in sorted(class_map.keys())]
    print(classification_report(y_test_encoded, y_pred_colonoscan, target_names=target_names, digits=4))

    # ===================================================================
    # SETTING GLOBAL STYLE UNTUK PLOTTING (AGAR SESUAI REQUEST)
    # ===================================================================
    FS_TITLE = 22
    FS_LABEL = 22
    FS_TICK  = 20
    LW_PLOT  = 3.0
    FS_LEGEND = 16

    # ===================================================================
    # TAHAP 6: PLOTTING GABUNGAN 1x2 (Separasi Fitur) - STYLED
    # ===================================================================
    
    print("📊 Plotting GABUNGAN 1x2 (Separasi Fitur) - Styled...")
    
    fig, axs = plt.subplots(1, 2, figsize=(20, 9)) # Ukuran diperbesar sedikit
    # fig.suptitle("Feature Separation Comparison (UMAP)", fontsize=FS_TITLE+2, fontweight='bold')
    
    # --- Plot 1: UMAP on Raw Pixels ---
    df_plot_raw = pd.DataFrame(X_test_umap_raw_pixels, columns=['UMAP0', 'UMAP1'])
    df_plot_raw['MES Class'] = [class_map[i] for i in y_test_encoded]
    
    sns.scatterplot(
        data=df_plot_raw, x='UMAP0', y='UMAP1', 
        hue='MES Class', hue_order=class_names,
        palette=sns.color_palette("bright", n_colors=len(class_names)),
        s=80, alpha=0.9, ax=axs[0], legend=False
    )
    
    axs[0].set_title("UMAP on Raw Pixels", fontsize=FS_TITLE, fontweight='bold')
    axs[0].set_xlabel("UMAP Dimension 1", fontsize=FS_LABEL, fontweight='bold')
    axs[0].set_ylabel("UMAP Dimension 2", fontsize=FS_LABEL, fontweight='bold')
    axs[0].tick_params(axis='both', which='major', labelsize=FS_TICK)
    axs[0].grid(True, alpha=0.3)

    # --- Plot 2: UMAP on Embeddings ---
    if X_test_umap_hybrid_embeddings is not None:
        df_plot_embed = pd.DataFrame(X_test_umap_hybrid_embeddings, columns=['UMAP0', 'UMAP1'])
        df_plot_embed['MES Class'] = [class_map[i] for i in y_test_encoded]
        
        sns.scatterplot(
            data=df_plot_embed, x='UMAP0', y='UMAP1',
            hue='MES Class', hue_order=class_names,
            palette=sns.color_palette("bright", n_colors=len(class_names)),
            s=80, alpha=0.9, ax=axs[1]
        )
        axs[1].set_title("UMAP on Hybrid Embeddings", fontsize=FS_TITLE, fontweight='bold')
        axs[1].set_xlabel("UMAP Dimension 1", fontsize=FS_LABEL, fontweight='bold')
        axs[1].set_ylabel("UMAP Dimension 2", fontsize=FS_LABEL, fontweight='bold')
        axs[1].tick_params(axis='both', which='major', labelsize=FS_TICK)
        axs[1].grid(True, alpha=0.3)
        
        # Legend Customization
        axs[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=FS_LEGEND, 
                      frameon=True, edgecolor='black', fancybox=True)
    else:
        axs[1].text(0.5, 0.5, "Embedding Extraction Failed", 
                    ha='center', va='center', fontsize=FS_TITLE)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(SAVE_DIR, "plot_separasi_GABUNGAN_1x2_Styled.png"), dpi=300)
    plt.close(fig)

    # ===================================================================
    # TAHAP 7: PLOTTING GABUNGAN 1x3 (t-SNE Akurasi) - STYLED
    # ===================================================================

    print("\n📊 Plotting Akurasi 1x3 (t-SNE) - Styled...")

    tsne_reducer = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, init='pca')
    X_tsne = tsne_reducer.fit_transform(X_feat_test_scaled)

    df_plot = pd.DataFrame(X_tsne, columns=['TSNE0', 'TSNE1'])
    df_plot['True'] = [class_map[i] for i in y_test_encoded]
    df_plot['Hybrid'] = [class_map[i] for i in y_pred_hybrid_selector]
    df_plot['ColonoScan'] = [class_map[i] for i in y_pred_colonoscan]

    fig_acc, axs_acc = plt.subplots(1, 3, figsize=(28, 9)) # Diperlebar
    # fig_acc.suptitle("Prediction Accuracy Visualization (t-SNE)", fontsize=FS_TITLE+2, fontweight='bold')

    titles = ["(a) Ground Truth", "(b) Hybrid Selector", "(c) ColonoScan Agent"]
    cols = ['True', 'Hybrid', 'ColonoScan']

    for i in range(3):
        sns.scatterplot(
            data=df_plot, x='TSNE0', y='TSNE1', 
            hue=cols[i], hue_order=class_names, 
            palette=sns.color_palette("bright", n_colors=len(class_names)), 
            s=80, alpha=0.9, ax=axs_acc[i], 
            legend=(i == 2)
        )
        axs_acc[i].set_title(titles[i], fontsize=FS_TITLE, fontweight='bold')
        axs_acc[i].set_xlabel("t-SNE Dimension 1", fontsize=FS_LABEL, fontweight='bold')
        axs_acc[i].set_ylabel("t-SNE Dimension 2", fontsize=FS_LABEL, fontweight='bold')
        axs_acc[i].tick_params(axis='both', which='major', labelsize=FS_TICK)
        axs_acc[i].grid(True, alpha=0.3)

    axs_acc[2].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=FS_LEGEND, 
                      frameon=True, edgecolor='black', fancybox=True)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(SAVE_DIR, "plot_akurasi_GABUNGAN_1x3_Styled.png"), dpi=300)
    plt.close(fig_acc)

    # ===================================================================
    # TAHAP 8: PLOTTING ROC CURVE - STYLED (SESUAI REQUEST)
    # ===================================================================

    print("\n📊 Plotting ROC Curves - Styled...")
    
    n_classes = len(class_names)
    y_true_bin = label_binarize(y_test_encoded, classes=range(n_classes))

    fpr = {"hybrid": dict(), "agent": dict()}
    tpr = {"hybrid": dict(), "agent": dict()}
    roc_auc = {"hybrid": dict(), "agent": dict()}

    for i in range(n_classes):
        fpr["hybrid"][i], tpr["hybrid"][i], _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc["hybrid"][i] = auc(fpr["hybrid"][i], tpr["hybrid"][i])
        fpr["agent"][i], tpr["agent"][i], _ = roc_curve(y_true_bin[:, i], y_proba_agent[:, i])
        roc_auc["agent"][i] = auc(fpr["agent"][i], tpr["agent"][i])

    fig_roc, axs_roc = plt.subplots(2, 2, figsize=(18, 16))
    # fig_roc.suptitle('ROC Curves Comparison', fontsize=FS_TITLE+2, fontweight='bold')

    hybrid_color = 'darkorange'
    agent_color = 'cornflowerblue'

    for i, ax in enumerate(axs_roc.flatten()):
        if i < n_classes:
            cls_name = class_names[i]
            
            # Plot Hybrid Line (Tebal)
            ax.plot(fpr["hybrid"][i], tpr["hybrid"][i], 
                    color=hybrid_color, lw=LW_PLOT, 
                    label=f'Hybrid (AUC={roc_auc["hybrid"][i]:0.3f})')
            
            # Plot Agent Line (Tebal)
            ax.plot(fpr["agent"][i], tpr["agent"][i], 
                    color=agent_color, lw=LW_PLOT, 
                    label=f'ColonoScan (AUC={roc_auc["agent"][i]:0.3f})')
            
            # Chance Line
            ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance')
            
            # Style Sumbu & Label
            ax.set_xlim([-0.02, 1.02])
            ax.set_ylim([-0.02, 1.02])
            ax.set_xlabel('False Positive Rate', fontsize=FS_LABEL, fontweight='bold')
            ax.set_ylabel('True Positive Rate', fontsize=FS_LABEL, fontweight='bold')
            ax.set_title(f'ROC: {cls_name}', fontsize=FS_TITLE, fontweight='bold')
            
            # Style Ticks (Angka Axis) - Ukuran 20
            ax.tick_params(axis='both', which='major', labelsize=FS_TICK)
            
            # Style Grid & Legend
            ax.grid(alpha=0.3)
            ax.legend(loc="lower right", fontsize=FS_LEGEND, 
                      frameon=True, edgecolor='black', fancybox=True)
        else:
            ax.axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(SAVE_DIR, "plot_ROC_curve_Styled.png"), dpi=300)
    plt.close(fig_roc)

    print(f"🎉🎉🎉 SEMUA PLOT SELESAI DENGAN STYLE BARU! Cek folder: {SAVE_DIR}")

except Exception as e:
    import traceback
    print("❌ Terjadi error:", e)
    traceback.print_exc()

In [ ]:
# --- IMPORTS WAJIB UNTUK EVALUASI ---
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import csv # Karena dipakai di dalam class DataDirTMCUCM


import csv, os
from collections import namedtuple

# BC prefix => borrowed code

class BC_FeatureExtractor:

    @staticmethod
    def _wavelet_stats(subband):
        return [
            np.mean(subband),
            np.std(subband),
            np.var(subband),
            scipy.stats.entropy(np.abs(subband.flatten()) + 1e-6)
        ]

    @classmethod
    def extract_wavelet_stats(cls, image, wavelet='db1'):
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        coeffs2 = pywt.dwt2(gray, wavelet)
        LL, (LH, HL, HH) = coeffs2
        features = list()
        for band in [LL, LH, HL, HH]:
            features.extend(cls._wavelet_stats(band))
        hh_energy = np.sum(np.square(HH))
        features.append(hh_energy)
        return features

    @staticmethod
    def extract_glcm_features_extended(image):
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        angles = [0, np.pi/4, np.pi/2]
        glcm = graycomatrix(gray, distances=[5], angles=angles, levels=256, symmetric=True, normed=True)
        contrast = graycoprops(glcm, 'contrast').mean()
        dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
        homogeneity = graycoprops(glcm, 'homogeneity').mean()
        return [contrast, dissimilarity, homogeneity]



class BaseDataDir:
        
    def _assert_files(self):
        discard = list()
        for i, f in enumerate(self.files):
            if not os.path.exists(f):
                print(f'warning: cannot resolve file {f}')
                discard.append(i)
        for i in reversed(discard):
            self.files.pop(i)

    def get_label(self, i):
        path = self.files[i]
        return self.get_label_from_path(path)
    
    def load(self, i):
        path = self.files[i]
        label = self.get_label_from_path(path)
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # img = img[30:430, 200:550]
        img = cv2.resize(img, IMG_SIZE)
        features = list()
        features.extend(BC_FeatureExtractor.extract_wavelet_stats(img, wavelet=WAVELET))
        features.extend(BC_FeatureExtractor.extract_glcm_features_extended(img))
        return img, features, label

    def imshows4(self, indices):
        fig, axs = plt.subplots(2, 2, figsize=(8, 4*2))
        for n, i in enumerate(indices):
            img, _, _ = self.load(i)
            axs[n//2, n%2].imshow(img)
            # axs[n//2, n%2].set_title(self.files[i])
        return fig

class DataDirTMCUCM(BaseDataDir):
    nt_labels = namedtuple('Label', ['MES','train_test'])

    def __init__(self, path, ignore_augment=True):
        self.path = path
        self.labels = dict()
        self.files = list()
        self.nfiles = 0
        self.train_indices = list()
        self.test_indices = list()

        self._labels_train_csv_path = f'{path}/train.txt'
        if not os.path.exists(self._labels_train_csv_path):
            raise FileNotFoundError('cant resolve labels csv file')
        with open(self._labels_train_csv_path) as fp:
            reader = csv.reader(fp, delimiter=' ')
            for filepath, mes in reader:
                if ignore_augment and 'augment' in filepath: continue
                filename = '/'.join(filepath.split('/')[-2:])
                self.labels[filename] = self.nt_labels(f'MES{int(mes)}', 0)
                self.files.append(f'{self.path}/{filename}')
                self.train_indices.append(len(self.files)-1)

        self._labels_test_csv_path = f'{path}/test.txt'
        if not os.path.exists(self._labels_test_csv_path):
            raise FileNotFoundError('cant resolve labels csv file')
        with open(self._labels_test_csv_path) as fp:
            reader = csv.reader(fp, delimiter=' ')
            for filepath, mes in reader:
                if ignore_augment and 'augment' in filepath: continue
                filename = '/'.join(filepath.split('/')[-2:])
                self.labels[filename] = self.nt_labels(f'MES{int(mes)}', 1)
                self.files.append(f'{self.path}/{filename}')
                self.test_indices.append(len(self.files)-1)
        self._assert_files()
        self.nfiles = len(self.files)
        print(f'found {self.nfiles} files')
        self.train_indices = np.array(self.train_indices)
        self.test_indices = np.array(self.test_indices)

    def get_label_from_path(self, path):
        filename = '/'.join(path.split('/')[-2:])
        return self.labels[filename]

    def load_indices(self, indices):
        imgs = list()
        features = list()
        labels = list()
        for i in tqdm.auto.tqdm(indices):
            img, feature, label = self.load(i)
            imgs.append(img)
            features.append(feature)
            labels.append(label[0])
        return np.array(imgs), np.array(features), np.array(labels)

def evaluate_using_dataloader(datadir_obj, mode='test'):
    """
    Fungsi ini menerima objek DataDirTMCUCM, mengambil file path dari list-nya,
    lalu mengirimnya ke pipeline prediksi kita (run_step_1 -> run_step_2).
    """
    
    print(f"\n🚀 MEMULAI EVALUASI VIA DATALOADER (Mode: {mode.upper()})")
    
    # 1. Tentukan indices mana yang mau dites (Train atau Test)
    if mode == 'test':
        indices = datadir_obj.test_indices
        print(f"📂 Menggunakan Split TEST: {len(indices)} gambar")
    else:
        indices = datadir_obj.train_indices
        print(f"📂 Menggunakan Split TRAIN: {len(indices)} gambar")

    y_true = []
    y_pred = []
    
    # 2. Loop menggunakan indices dari DataDir
    for i in tqdm(indices, desc="Evaluasi Berjalan"):
        try:
            # A. Dapatkan Path File Asli
            # files[i] berisi path absolut/relatif yang sudah dibangun oleh class
            file_path = datadir_obj.files[i]
            
            # B. Dapatkan Label Asli
            # Class kamu mengembalikan namedtuple: Label(MES='MES0', train_test=...)
            label_struct = datadir_obj.get_label_from_path(file_path)
            label_str = label_struct.MES  # Isinya 'MES0', 'MES1', dst.
            
            # Konversi 'MES0' -> 0 (integer)
            true_label = int(label_str.replace('MES', '')) 
            
            # C. Lakukan Prediksi (Pakai Pipeline 2-Langkah kita yang tadi)
            # Kita kirim file_path ke fungsi run_step_1 yang sudah kita buat sebelumnya
            step_1_out = run_step_1_keras(file_path)
            
            if step_1_out is None: 
                continue # Skip jika gambar rusak/gagal load
                
            final_proba = run_step_2_lgbm(step_1_out)
            
            if final_proba is None:
                continue

            # Ambil kelas dengan probabilitas tertinggi
            pred_label = int(np.argmax(final_proba))
            
            # D. Simpan hasil
            y_true.append(true_label)
            y_pred.append(pred_label)
            
        except Exception as e:
            print(f"\n❌ Error pada index {i}: {e}")
            continue

    # --- HASIL AKHIR ---
    print("\n" + "="*50)
    print(f"📊 HASIL EVALUASI TMC-UCM ({mode.upper()} SET)")
    print("="*50)

    if len(y_true) == 0:
        print("❌ Tidak ada data yang berhasil diproses.")
        return

    # Hitung Akurasi
    acc = accuracy_score(y_true, y_pred)
    print(f"✅ Total Gambar: {len(y_true)}")
    print(f"✅ Akurasi: {acc:.4f} ({acc*100:.2f}%)")

    # Classification Report
    target_names = ['MES 0', 'MES 1', 'MES 2', 'MES 3']
    print("\n📝 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=target_names))

    # Confusion Matrix Plot
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=target_names, yticklabels=target_names)
    plt.xlabel('Prediksi Model')
    plt.ylabel('Ground Truth (Label Asli)')
    plt.title(f'Confusion Matrix - TMC UCM {mode.upper()} (Acc: {acc:.2%})')
    plt.show()

# === CONTOH CARA PAKAI DI MAIN ===
if __name__ == "__main__":
    
    # 1. Inisialisasi Class DataDirTMCUCM punya kamu
    # Pastikan path ini mengarah ke folder yang ada train.txt dan test.txt
    print("Membaca struktur dataset...")
    datadir_tmcucm = DataDirTMCUCM('./Dataset/TMC-UCM') 
    
    # 2. Jalankan Evaluasi pada data TEST
    evaluate_using_dataloader(datadir_tmcucm, mode='test')

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import joblib
import pywt
import scipy.stats
import matplotlib.pyplot as plt
import matplotlib as mpl
from skimage.feature import graycomatrix, graycoprops
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Concatenate, BatchNormalization, Activation
import lightgbm as lgb

# ==========================================
# 1. KONFIGURASI & LIST GAMBAR
# ==========================================
MODEL_DIR = "./Result/CrossDataset/TrainingMixedTesting1/"
IMG_SIZE = (224, 224)
WAVELET = 'db1'

# Daftar Gambar yang akan diproses (Tambahkan 1 gambar lagi di sini)
TARGET_IMAGES = [
    "./Result/CrossDataset/TrainingMixedTesting1/Figure 7 MES1.jpg",
    "./Result/CrossDataset/TrainingMixedTesting1/Figure 7_MES1_choice1.jpg",
    "./Result/CrossDataset/TrainingMixedTesting1/Figure 7_MES1_choice2.jpg",
    "./Result/CrossDataset/TrainingMixedTesting1/Figure 7_MES2_choice1.jpg",
    "./Result/CrossDataset/TrainingMixedTesting1/Figure 7_MES2_choice2.jpg",
    "./Result/CrossDataset/TrainingMixedTesting1/Figure 7_MES2_choice3.jpg" # Ganti dengan path gambar ke-6 Anda
]

LABELS = ['MES 0', 'MES 1', 'MES 2', 'MES 3']

# ==========================================
# 2. LOAD MODELS & DEFINITIONS (Sama persis, tidak ada perubahan)
# ==========================================
def GroupConv2D(filters, kernel_size, strides=(1, 1), padding='same', groups=3):
    def layer(x): return tf.keras.layers.Conv2D(filters, kernel_size, strides=strides, padding=padding)(x)
    return layer
def SE2MaxPooling2D(pool_size=(2, 2)):
    def layer(x): return tf.keras.layers.MaxPooling2D(pool_size=pool_size)(x)
    return layer
def SE2LiftingLayer(x):
    def layer(x): return x
    return layer
def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred): return tf.reduce_mean(y_pred)
    return loss

print("📦 Loading System...")
try:
    custom_objs = {'loss': focal_loss(), 'GroupConv2D': GroupConv2D, 'SE2MaxPooling2D': SE2MaxPooling2D, 'SE2LiftingLayer': SE2LiftingLayer}
    hybrid_model = load_model(os.path.join(MODEL_DIR, "TryFindingBestModel.h5"), custom_objects=custom_objs)
    scaler_20 = joblib.load(os.path.join(MODEL_DIR, "scaler_handcrafted_20.pkl"))
    umap_model = joblib.load(os.path.join(MODEL_DIR, "umap_model_mixed.pkl"))
    lgbm_agent = lgb.Booster(model_file=os.path.join(MODEL_DIR, "feedback_agent.txt"))
    scaler_23 = joblib.load(os.path.join(MODEL_DIR, "scaler_agent.pkl"))
    print("✅ System Loaded Successfully.")
except Exception as e:
    print(f"❌ Error Load: {e}")

# ==========================================
# 3. HELPER FUNCTIONS (Sama persis, tidak ada perubahan)
# ==========================================
def extract_features(img_rgb):
    if len(img_rgb.shape) == 3: gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    else: gray = img_rgb
    coeffs = pywt.dwt2(gray, WAVELET)
    LL, (LH, HL, HH) = coeffs
    def s(sub): return [np.mean(sub), np.std(sub), np.var(sub), scipy.stats.entropy(np.abs(sub.flatten())+1e-6)]
    feats = []
    for b in [LL, LH, HL, HH]: feats.extend(s(b))
    feats.append(np.sum(np.square(HH)))
    g_u8 = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    glcm = graycomatrix(g_u8, [5], [0, np.pi/4, np.pi/2], 256, symmetric=True, normed=True)
    feats += [graycoprops(glcm, p).mean() for p in ['contrast', 'dissimilarity', 'homogeneity']]
    return feats

def get_model_inputs(img_rgb):
    try: cr = img_rgb[30:430, 200:550]
    except: cr = img_rgb
    re = cv2.resize(cr, IMG_SIZE)
    
    ft = extract_features(re)
    ft_s = scaler_20.transform(np.array(ft).reshape(1, -1))
    um = umap_model.transform(ft_s)
    inp = np.expand_dims(re.astype(np.float32)/255.0, axis=0)
    
    return re, inp, ft_s, um

# ==========================================
# 4. CORE HEATMAP GENERATOR (Sama persis, tidak ada perubahan)
# ==========================================
def generate_attention_map(img_path, mode='lgbm', patch_size=35, stride=10):
    img_raw = cv2.imread(img_path)
    if img_raw is None: return None, None, None, 0.0
    img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
    
    re, inp, ft_s, um = get_model_inputs(img_rgb)
    k_pred = hybrid_model.predict([inp, ft_s, um], verbose=0)
    cnn_conf = np.max(k_pred)
    target_idx = np.argmax(k_pred[0]) 
    
    if mode == 'lgbm':
        l_in = np.concatenate(([cnn_conf], um[0], ft_s[0])).reshape(1, -1)
        l_pred = lgbm_agent.predict(scaler_23.transform(l_in))[0]
        if isinstance(l_pred, np.ndarray): 
            base_score = np.max(l_pred)
            target_idx = np.argmax(l_pred)
        else: 
            base_score = l_pred if l_pred > 0.5 else (1-l_pred)
            target_idx = int(l_pred > 0.5)
    else:
        base_score = k_pred[0][target_idx]

    w, h = IMG_SIZE
    heatmap = np.zeros((w, h))
    base_img = re.copy()
    masks, coords = [], []
    for y in range(0, h-patch_size+1, stride):
        for x in range(0, w-patch_size+1, stride):
            m = base_img.copy()
            m[y:y+patch_size, x:x+patch_size, :] = 128
            masks.append(m)
            coords.append((x,y))
            
    for i, (x,y) in enumerate(coords):
        masked = masks[i]
        if mode == 'cnn':
            m_inp = np.expand_dims(masked.astype(np.float32)/255.0, axis=0)
            pred = hybrid_model.predict([m_inp, ft_s, um], verbose=0)[0][target_idx]
        else:
            f_new = extract_features(masked)
            fs_new = scaler_20.transform(np.array(f_new).reshape(1, -1))
            u_new = umap_model.transform(fs_new)
            i_new = np.expand_dims(masked.astype(np.float32)/255.0, axis=0)
            c_new = np.max(hybrid_model.predict([i_new, fs_new, u_new], verbose=0))
            ln_new = np.concatenate(([c_new], u_new[0], fs_new[0])).reshape(1, -1)
            lp_new = lgbm_agent.predict(scaler_23.transform(ln_new))[0]
            if isinstance(lp_new, np.ndarray): pred = lp_new[target_idx]
            else: pred = lp_new if target_idx==1 else (1-lp_new)
        heatmap[y:y+patch_size, x:x+patch_size] = base_score - pred

    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) > 0: heatmap /= np.max(heatmap)
    heatmap_smooth = cv2.GaussianBlur(heatmap, (25, 25), 0)
    hm_u8 = np.uint8(255 * heatmap_smooth)
    hm_c = cv2.applyColorMap(hm_u8, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(re, 0.6, hm_c, 0.4, 0)
    return overlay, target_idx, base_score, re

# ==========================================
# 5. MAIN LOOP & PLOTTING DENGAN 6 BARIS & TAMPILAN RAPAT
# ==========================================
num_images = len(TARGET_IMAGES)
print(f"🚀 Processing {num_images} Images...")

results = []
for idx, img_path in enumerate(TARGET_IMAGES):
    filename = os.path.basename(img_path)
    print(f"[{idx+1}/{num_images}] Analyzing: {filename} ...")
    if not os.path.exists(img_path):
        print(f"⚠️ Skipping {filename} (File Not Found)")
        continue
    ov_cnn, _, _, _ = generate_attention_map(img_path, mode='cnn')
    ov_lgbm, pred_cls, conf, orig_img = generate_attention_map(img_path, mode='lgbm')
    results.append({
        'orig': orig_img, 'cnn': ov_cnn, 'lgbm': ov_lgbm,
        'label': LABELS[pred_cls], 'fname': filename
    })

# === PENGATURAN FIGURE AGAR RAPAT ===
# Ubah tinggi per baris lebih kecil agar lebih padat
row_height = 2.5 # Nilai ini diperkecil
fig_height = len(results) * row_height
print("🎨 Creating Final Figure...")

# figsize harus cukup lebar untuk 4 kolom, dan tinggi menyesuaikan jumlah baris
fig, axes = plt.subplots(len(results), 4, figsize=(12, fig_height)) # figsize lebar diperkecil
if len(results) == 1: axes = np.expand_dims(axes, axis=0)

# Loop Plotting Utama (Tanpa Title per Subplot agar Rapat)
for i, res in enumerate(results):
    axes[i, 0].imshow(res['orig'])
    axes[i, 0].axis('off') # Tidak ada axis, tidak ada title

    axes[i, 1].imshow(res['cnn'])
    axes[i, 1].axis('off')

    axes[i, 2].imshow(res['lgbm'])
    axes[i, 2].axis('off')

    axes[i, 3].axis('off')
    # Teks Label dibuat besar dan bold di sebelah kanan
    axes[i, 3].text(0.1, 0.5, res['label'], ha='left', va='center', 
                    fontsize=20, weight='bold', color='black')

# --- BAGIAN COLORBAR DI KANAN (Menggunakan Gambar sebagai Referensi) ---

# 1. Atur layout agar subplot sangat rapat
# wspace dan hspace yang kecil (mendekati 0) membuat jarak antar subplot hilang
plt.subplots_adjust(left=0.05, right=0.88, top=0.98, bottom=0.02, wspace=0.05, hspace=0.05)

# 2. Dapatkan posisi total area gambar (kolom 1, 2, 3) untuk menempatkan colorbar di kanannya
# Ambil posisi dari axis paling kiri atas dan axis kolom ke-3 paling bawah
pos_top_left = axes[0, 0].get_position()
pos_bottom_right_img = axes[-1, 2].get_position() # Kolom ke-2 (indeks 2) adalah kolom gambar terakhir

cbar_height = pos_top_left.ymax - pos_bottom_right_img.ymin
cbar_bottom = pos_bottom_right_img.ymin

# 3. Buat axis manual untuk colorbar di sebelah kanan kolom gambar 'Stage 2'
# Koordinat [left, bottom, width, height]
cbar_ax = fig.add_axes([0.90, cbar_bottom, 0.02, cbar_height])

# 4. Buat mappable untuk colormap 'jet'
norm = mpl.colors.Normalize(vmin=0, vmax=1)
mappable = mpl.cm.ScalarMappable(norm=norm, cmap='jet')

# 5. Gambar colorbar vertikal
cb = fig.colorbar(mappable, cax=cbar_ax, orientation='vertical')
# Tidak perlu label 'Attention Level' agar mirip gambar referensi
# cb.set_label('Attention Level', fontsize=12, labelpad=10) 

# Hapus ticks agar bersih seperti di gambar referensi (opsional)
cb.set_ticks([]) 

save_path = f"Final_Result_visualisation_test.jpg"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"✅ DONE! Saved to {save_path}")
plt.show()

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import joblib
import pywt
import scipy.stats
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable 
from skimage.feature import graycomatrix, graycoprops
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Concatenate, BatchNormalization, Activation
import lightgbm as lgb

# ==========================================
# 1. KONFIGURASI FOLDER & DATASET
# ==========================================
MODEL_DIR = "./Result/CrossDataset/TrainingMixedTesting1/" 
IMG_SIZE = (224, 224)
WAVELET = 'db1'
BATCH_SIZE = 5  # <-- VISUALISASI PER 5 GAMBAR

# ---------------------------------------------------------
# 👇 GANTI NAMA FOLDER DI SINI UNTUK EVALUASI PER KELAS 👇
# ---------------------------------------------------------
SOURCE_FOLDER = "./MES classification_20250724/MES3" 

print(f"📂 Reading images from: {SOURCE_FOLDER}")

# --- SETUP FOLDER PENYIMPANAN KHUSUS ---
folder_name_only = os.path.basename(SOURCE_FOLDER.rstrip('/'))
OUTPUT_SAVE_DIR = os.path.join(MODEL_DIR, f"EVAL_RESULTS_{folder_name_only}")

# Buat folder jika belum ada
if not os.path.exists(OUTPUT_SAVE_DIR):
    os.makedirs(OUTPUT_SAVE_DIR)
    print(f"📁 Created new directory for results: {OUTPUT_SAVE_DIR}")
else:
    print(f"📁 Saving results to existing directory: {OUTPUT_SAVE_DIR}")

# Ambil semua file gambar
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
try:
    file_list = sorted([f for f in os.listdir(SOURCE_FOLDER) if f.lower().endswith(valid_extensions)])
    TARGET_IMAGES = [os.path.join(SOURCE_FOLDER, f) for f in file_list]
    print(f"✅ Found {len(TARGET_IMAGES)} images.")
except FileNotFoundError:
    print(f"❌ Error: Folder '{SOURCE_FOLDER}' tidak ditemukan.")
    TARGET_IMAGES = []

LABELS = ['MES 0', 'MES 1', 'MES 2', 'MES 3']

# ==========================================
# 2. LOAD MODELS & DEFINITIONS
# ==========================================
def GroupConv2D(filters, kernel_size, strides=(1, 1), padding='same', groups=3):
    def layer(x): return tf.keras.layers.Conv2D(filters, kernel_size, strides=strides, padding=padding)(x)
    return layer

def SE2MaxPooling2D(pool_size=(2, 2)):
    def layer(x): return tf.keras.layers.MaxPooling2D(pool_size=pool_size)(x)
    return layer

def SE2LiftingLayer(x):
    def layer(x): return x
    return layer

def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred): return tf.reduce_mean(y_pred)
    return loss

print("📦 Loading System...")
try:
    custom_objs = {'loss': focal_loss(), 'GroupConv2D': GroupConv2D, 'SE2MaxPooling2D': SE2MaxPooling2D, 'SE2LiftingLayer': SE2LiftingLayer}
    hybrid_model = load_model(os.path.join(MODEL_DIR, "TryFindingBestModel.h5"), custom_objects=custom_objs)
    scaler_20 = joblib.load(os.path.join(MODEL_DIR, "scaler_handcrafted_20.pkl"))
    umap_model = joblib.load(os.path.join(MODEL_DIR, "umap_model_mixed.pkl"))
    lgbm_agent = lgb.Booster(model_file=os.path.join(MODEL_DIR, "feedback_agent.txt"))
    scaler_23 = joblib.load(os.path.join(MODEL_DIR, "scaler_agent.pkl"))
    print("✅ System Loaded Successfully.")
except Exception as e:
    print(f"❌ Error Load: {e}")

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def extract_features(img_rgb):
    if len(img_rgb.shape) == 3: gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    else: gray = img_rgb
    coeffs = pywt.dwt2(gray, WAVELET)
    LL, (LH, HL, HH) = coeffs
    def s(sub): return [np.mean(sub), np.std(sub), np.var(sub), scipy.stats.entropy(np.abs(sub.flatten())+1e-6)]
    feats = []
    for b in [LL, LH, HL, HH]: feats.extend(s(b))
    feats.append(np.sum(np.square(HH)))
    g_u8 = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    glcm = graycomatrix(g_u8, [5], [0, np.pi/4, np.pi/2], 256, symmetric=True, normed=True)
    feats += [graycoprops(glcm, p).mean() for p in ['contrast', 'dissimilarity', 'homogeneity']]
    return feats

def get_model_inputs(img_rgb):
    try: cr = img_rgb[30:430, 200:550]
    except: cr = img_rgb
    re = cv2.resize(cr, IMG_SIZE)
    ft = extract_features(re)
    ft_s = scaler_20.transform(np.array(ft).reshape(1, -1))
    um = umap_model.transform(ft_s)
    inp = np.expand_dims(re.astype(np.float32)/255.0, axis=0)
    return re, inp, ft_s, um

# ==========================================
# 4. CORE HEATMAP GENERATOR
# ==========================================
def generate_attention_map(img_path, mode='lgbm', patch_size=35, stride=10):
    img_raw = cv2.imread(img_path)
    if img_raw is None: return None, None, None, 0.0
    img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
    
    # 1. Baseline
    re, inp, ft_s, um = get_model_inputs(img_rgb)
    k_pred = hybrid_model.predict([inp, ft_s, um], verbose=0)
    cnn_conf = np.max(k_pred)
    target_idx = np.argmax(k_pred[0]) 
    
    if mode == 'lgbm':
        l_in = np.concatenate(([cnn_conf], um[0], ft_s[0])).reshape(1, -1)
        l_pred = lgbm_agent.predict(scaler_23.transform(l_in))[0]
        if isinstance(l_pred, np.ndarray): 
            base_score = np.max(l_pred)
            target_idx = np.argmax(l_pred)
        else: 
            base_score = l_pred if l_pred > 0.5 else (1-l_pred)
            target_idx = int(l_pred > 0.5)
    else:
        base_score = k_pred[0][target_idx]

    # 2. Scanning
    w, h = IMG_SIZE
    heatmap = np.zeros((w, h))
    base_img = re.copy()
    masks, coords = [], []
    for y in range(0, h-patch_size+1, stride):
        for x in range(0, w-patch_size+1, stride):
            m = base_img.copy()
            m[y:y+patch_size, x:x+patch_size, :] = 128
            masks.append(m)
            coords.append((x,y))
            
    # Process Masks
    for i, (x,y) in enumerate(coords):
        masked = masks[i]
        if mode == 'cnn':
            m_inp = np.expand_dims(masked.astype(np.float32)/255.0, axis=0)
            pred = hybrid_model.predict([m_inp, ft_s, um], verbose=0)[0][target_idx]
        else:
            f_new = extract_features(masked)
            fs_new = scaler_20.transform(np.array(f_new).reshape(1, -1))
            u_new = umap_model.transform(fs_new)
            i_new = np.expand_dims(masked.astype(np.float32)/255.0, axis=0)
            c_new = np.max(hybrid_model.predict([i_new, fs_new, u_new], verbose=0))
            ln_new = np.concatenate(([c_new], u_new[0], fs_new[0])).reshape(1, -1)
            lp_new = lgbm_agent.predict(scaler_23.transform(ln_new))[0]
            if isinstance(lp_new, np.ndarray): pred = lp_new[target_idx]
            else: pred = lp_new if target_idx==1 else (1-lp_new)
        heatmap[y:y+patch_size, x:x+patch_size] = base_score - pred

    # 3. Smoothing
    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) > 0: heatmap /= np.max(heatmap)
    heatmap_smooth = cv2.GaussianBlur(heatmap, (25, 25), 0)
    
    hm_u8 = np.uint8(255 * heatmap_smooth)
    hm_c = cv2.applyColorMap(hm_u8, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(re, 0.6, hm_c, 0.4, 0)
    return overlay, target_idx, base_score, re

# ==========================================
# 5. FUNGSI PLOTTING BATCH (SAVE KE SUB-FOLDER)
# ==========================================
def plot_batch_results(batch_results, batch_idx, output_dir):
    if not batch_results: return

    row_height = 4
    fig_height = len(batch_results) * row_height
    
    print(f"📊 Plotting Batch {batch_idx} ({len(batch_results)} images)...")
    
    fig, axes = plt.subplots(len(batch_results), 4, figsize=(22, fig_height)) 
    if len(batch_results) == 1: axes = np.expand_dims(axes, axis=0)

    norm = mpl.colors.Normalize(vmin=0, vmax=1)
    cmap = plt.get_cmap('jet')
    mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)

    for i, res in enumerate(batch_results):
        # Col 0
        axes[i, 0].imshow(res['orig'])
        axes[i, 0].axis('off')
        axes[i, 0].text(10, 200, res['fname'], color='white', backgroundcolor='black', fontsize=12)
        if i == 0: axes[i, 0].set_title("Original Image", fontsize=14, weight='bold')
        
        # Col 1
        im1 = axes[i, 1].imshow(res['cnn'])
        axes[i, 1].axis('off')
        if i == 0: axes[i, 1].set_title("Stage 1: CNN Focus", fontsize=14, weight='bold')
        divider1 = make_axes_locatable(axes[i, 1])
        cax1 = divider1.append_axes("right", size="5%", pad=0.1)
        cb1 = plt.colorbar(mappable, cax=cax1)
        cb1.ax.tick_params(labelsize=20) 
        
        # Col 2
        im2 = axes[i, 2].imshow(res['lgbm'])
        axes[i, 2].axis('off')
        if i == 0: axes[i, 2].set_title("Stage 2: Agent Focus", fontsize=14, weight='bold')
        divider2 = make_axes_locatable(axes[i, 2])
        cax2 = divider2.append_axes("right", size="5%", pad=0.1)
        cb2 = plt.colorbar(mappable, cax=cax2)
        cb2.ax.tick_params(labelsize=20)

        # Col 3
        axes[i, 3].axis('off')
        axes[i, 3].text(0.5, 0.5, res['label'], ha='center', va='center', 
                        fontsize=24, weight='bold', color='darkblue')
        if i == 0: axes[i, 3].set_title("Prediction", fontsize=14, weight='bold')

    plt.tight_layout()
    
    # --- SIMPAN KE FOLDER KHUSUS ---
    filename_img = f"Batch_{batch_idx}.jpg"
    save_path = os.path.join(output_dir, filename_img)
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✅ BATCH {batch_idx} SAVED TO: {save_path}")
    
    # Tampilkan juga di layar (opsional, jika ingin melihat sekilas)
    plt.show()
    plt.close()

# ==========================================
# 6. MAIN EXECUTION
# ==========================================
num_images = len(TARGET_IMAGES)

if num_images == 0:
    print("⚠️ No images found! Check your folder path.")
else:
    print(f"🚀 Processing {num_images} Images (Batch Size: {BATCH_SIZE})...")
    print(f"📂 Output Folder: {OUTPUT_SAVE_DIR}")

    current_batch = []
    batch_counter = 1

    for idx, img_path in enumerate(TARGET_IMAGES):
        filename = os.path.basename(img_path)
        print(f"[{idx+1}/{num_images}] Analyzing: {filename} ...")
        
        ov_cnn, _, _, _ = generate_attention_map(img_path, mode='cnn')
        if ov_cnn is None: continue 
        ov_lgbm, pred_cls, conf, orig_img = generate_attention_map(img_path, mode='lgbm')
        
        current_batch.append({
            'orig': orig_img,
            'cnn': ov_cnn,
            'lgbm': ov_lgbm,
            'label': LABELS[pred_cls],
            'fname': filename
        })

        if len(current_batch) == BATCH_SIZE:
            plot_batch_results(current_batch, batch_counter, OUTPUT_SAVE_DIR)
            current_batch = [] 
            batch_counter += 1

    if len(current_batch) > 0:
        print(f"🧹 Processing remaining {len(current_batch)} images...")
        plot_batch_results(current_batch, batch_counter, OUTPUT_SAVE_DIR)

    print(f"\n🎉 ALL DONE! Check the folder: {OUTPUT_SAVE_DIR}")

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. KONFIGURASI
# ==========================================
IMG_SIZE = (224, 224)

# Ambil satu gambar saja dari list (Ganti index jika ingin gambar lain)
TARGET_IMAGE_PATH = "./Result/CrossDataset/TrainingMixedTesting1/test1.jpg" 

# Jika file diatas tidak ada, ganti dengan path absolut untuk testing
# TARGET_IMAGE_PATH = "path/to/your/image.jpg"

# ==========================================
# 2. FUNGSI ROTASI
# ==========================================
def rotate_image(image, angle):
    """Memutar gambar tanpa memotong sudut (tetap center)"""
    if angle == 0: return image
    
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    
    # M = Matrix Rotasi
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    
    # Lakukan rotasi
    rotated = cv2.warpAffine(image, M, (w, h))
    return rotated

# ==========================================
# 3. PROSES UTAMA
# ==========================================
if not os.path.exists(TARGET_IMAGE_PATH):
    print(f"❌ File tidak ditemukan: {TARGET_IMAGE_PATH}")
else:
    print(f"📸 Memproses: {os.path.basename(TARGET_IMAGE_PATH)}")
    
    # 1. Load Gambar
    img_raw = cv2.imread(TARGET_IMAGE_PATH)
    img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)

    # 2. Lakukan Cropping (Sesuai kodemu sebelumnya)
    try:
        # Koordinat crop dari kodemu: [30:430, 200:550]
        crop = img_rgb[30:430, 200:550]
    except Exception as e:
        print("Gagal crop, menggunakan gambar asli.")
        crop = img_rgb

    # 3. Resize ke 224x224
    img_final = cv2.resize(crop, IMG_SIZE)

    # ==========================================
    # 4. SETUP GRID ROTASI (3x3)
    # ==========================================
    # Kita definisikan list dictionary agar mirip dengan gambar referensi
    # Format: (Label, Sudut Rotasi)
    # Susunan Grid 3x3 (Baris demi baris)
    augmentations = [
        {"label": "Rotated +30°",  "angle": 30},
        {"label": "Rotated +60°",  "angle": 60}, 
        {"label": "Rotated -30°",  "angle": -30},
        
        {"label": "Original",      "angle": 0},   # Posisi Tengah Kiri (sesuai referensi)
        {"label": "Rotated +90°",  "angle": 90},
        {"label": "Rotated -135°", "angle": -135},
        
        {"label": "Rotated -60°",  "angle": -60},
        {"label": "Rotated +135°", "angle": 135},
        {"label": "Rotated -90°",  "angle": -90},
    ]

    # Buat Plot
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    plt.subplots_adjust(wspace=0.1, hspace=0.2) # Jarak antar gambar

    for i, data in enumerate(augmentations):
        row = i // 3
        col = i % 3
        
        # Ambil gambar hasil rotasi
        res_img = rotate_image(img_final, data["angle"])
        
        # Tampilkan di subplot
        axes[row, col].imshow(res_img)
        
        # Tambahkan Label di bawah gambar (seperti referensi)
        axes[row, col].text(0.5, -0.15, data["label"], 
                            transform=axes[row, col].transAxes, 
                            ha='center', fontsize=12, weight='bold', 
                            color='white', backgroundcolor='black')
        
        axes[row, col].axis('off') # Hilangkan axis angka

    plt.tight_layout()
    plt.savefig("Rotated_Grid_Result.jpg", dpi=300)
    print("✅ Gambar berhasil disimpan sebagai 'Rotated_Grid_Result.jpg'")
    plt.show()

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pywt
import scipy.stats
from skimage.feature import graycomatrix, graycoprops
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. KONFIGURASI
# ==========================================
DATASET_ROOT = "./MES Mixed Data"  # <-- Pastikan path ini benar
IMG_SIZE = (224, 224)
WAVELET = 'db1'

# ==========================================
# 2. FUNGSI EKSTRAKSI (SAMA SEPERTI SEBELUMNYA)
# ==========================================
def get_clinical_features(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, IMG_SIZE)
    gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)
    
    # Wavelet HH Energy
    coeffs = pywt.dwt2(gray, WAVELET)
    LL, (LH, HL, HH) = coeffs
    wavelet_energy = np.sum(np.square(HH))

    # GLCM Features
    if gray.max() > 1:
        gray_uint = gray.astype(np.uint8)
    else:
        gray_uint = (gray * 255).astype(np.uint8)
        
    glcm = graycomatrix(gray_uint, distances=[5], angles=[0, np.pi/4, np.pi/2], 
                        levels=256, symmetric=True, normed=True)

    contrast = graycoprops(glcm, 'contrast').mean()

    # Entropy Manual
    glcm_flat = glcm.flatten()
    glcm_flat = glcm_flat[glcm_flat > 0]
    entropy = -np.sum(glcm_flat * np.log(glcm_flat))

    return {
        "Wavelet HH Energy": wavelet_energy,
        "GLCM Contrast": contrast,
        "GLCM Entropy": entropy
    }

# ==========================================
# 3. MAIN LOOP
# ==========================================
def main():
    print(f"📂 Membaca dataset dari: {DATASET_ROOT}")
    data = []
    categories = ["MES0", "MES1", "MES2", "MES3"] 
    
    for label in categories:
        folder_candidates = [label, "class_" + label, label.replace("MES", "")]
        folder_path = None
        for candidate in folder_candidates:
            temp_path = os.path.join(DATASET_ROOT, candidate)
            if os.path.exists(temp_path):
                folder_path = temp_path
                break
        
        if not folder_path:
            print(f"⚠️ Warning: Folder {label} tidak ketemu. Skip.")
            continue
            
        print(f"   -> Processing {label}...")
        files = os.listdir(folder_path)
        
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                full_path = os.path.join(folder_path, f)
                feats = get_clinical_features(full_path)
                if feats:
                    row = {"Group": label}
                    row.update(feats)
                    data.append(row)

    if not data:
        print("❌ Error: Tidak ada data. Cek path folder.")
        return

    df = pd.DataFrame(data)
    print(f"✅ Selesai! Total {len(df)} gambar.")

    # ==========================================
    # 4. VISUALISASI "BIG & BOLD" (PUBLICATION READY)
    # ==========================================
    
    # Ukuran gambar diperbesar (Lebar 20, Tinggi 7)
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    sns.set_style("whitegrid")
    palette = sns.color_palette("viridis", 4) 
    
    # --- PENGATURAN FONT (Ubah angka di sini jika masih kurang besar) ---
    FONT_TITLE = 18       # Ukuran Judul Atas
    FONT_LABEL = 16       # Ukuran Label Sumbu (X/Y)
    FONT_TICK = 14        # Ukuran Angka/Teks di Sumbu (Index X & Y)
    
    # --- PLOT 1: WAVELET ENERGY ---
    df['Log Energy'] = np.log1p(df['Wavelet HH Energy'])
    sns.boxplot(x='Group', y='Log Energy', data=df, palette=palette, showfliers=False, order=categories, ax=axes[0])
    sns.stripplot(x='Group', y='Log Energy', data=df, color='black', alpha=0.15, jitter=True, size=4, order=categories, ax=axes[0])
    
    axes[0].set_title("Vascular Pattern Marker\n(Wavelet Energy)", fontsize=FONT_TITLE, fontweight='bold')
    axes[0].set_ylabel("Log(High-Frequency Energy)", fontsize=FONT_LABEL, fontweight='bold')
    axes[0].set_xlabel("Severity Score", fontsize=FONT_LABEL, fontweight='bold')
    axes[0].tick_params(axis='both', which='major', labelsize=FONT_TICK) # Membesarkan Index X dan Y

    # --- PLOT 2: GLCM ENTROPY ---
    sns.boxplot(x='Group', y='GLCM Entropy', data=df, palette=palette, showfliers=False, order=categories, ax=axes[1])
    sns.stripplot(x='Group', y='GLCM Entropy', data=df, color='black', alpha=0.15, jitter=True, size=4, order=categories, ax=axes[1])
    
    axes[1].set_title("Tissue Disorder Marker\n(GLCM Entropy)", fontsize=FONT_TITLE, fontweight='bold')
    axes[1].set_ylabel("Texture Entropy", fontsize=FONT_LABEL, fontweight='bold')
    axes[1].set_xlabel("Severity Score", fontsize=FONT_LABEL, fontweight='bold')
    axes[1].tick_params(axis='both', which='major', labelsize=FONT_TICK)

    # --- PLOT 3: GLCM CONTRAST ---
    sns.boxplot(x='Group', y='GLCM Contrast', data=df, palette=palette, showfliers=False, order=categories, ax=axes[2])
    sns.stripplot(x='Group', y='GLCM Contrast', data=df, color='black', alpha=0.15, jitter=True, size=4, order=categories, ax=axes[2])
    
    axes[2].set_title("Ulceration & Bleeding Marker\n(GLCM Contrast)", fontsize=FONT_TITLE, fontweight='bold')
    axes[2].set_ylabel("Texture Contrast", fontsize=FONT_LABEL, fontweight='bold')
    axes[2].set_xlabel("Severity Score", fontsize=FONT_LABEL, fontweight='bold')
    axes[2].tick_params(axis='both', which='major', labelsize=FONT_TICK)

    plt.tight_layout()
    save_name = "Manuscript_Evidence_BigFont.png"
    plt.savefig(save_name, dpi=300)
    print(f"💾 SUKSES! Gambar besar tersimpan sebagai: {save_name}")
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import joblib
import pywt
import scipy.stats
import matplotlib.pyplot as plt
import matplotlib as mpl
from skimage.feature import graycomatrix, graycoprops
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Concatenate, BatchNormalization, Activation
import lightgbm as lgb

# ==========================================
# 1. KONFIGURASI & LIST GAMBAR
# ==========================================
MODEL_DIR = "./Result/CrossDataset/TrainingMixedTesting1/"
IMG_SIZE = (224, 224)
WAVELET = 'db1'

# Daftar Gambar yang akan diproses (Tambahkan 1 gambar lagi di sini)
TARGET_IMAGES = [
    "./Result/CrossDataset/TrainingMixedTesting1/Figure 7_MES2_choice1.jpg",
    "./Result/CrossDataset/TrainingMixedTesting1/Figure 7_MES2_choice2.jpg",
    "./Result/CrossDataset/TrainingMixedTesting1/Figure 7_MES2_choice3.jpg”,
    "./Result/CrossDataset/TrainingMixedTesting1/class_3_2.jpg",
    "./MES classification_20250724/MES3/1487808_20211208_ES_1_47_29.jpg",
    "./MES classification_20250724/MES3/0987532_20201105_ES_1_4_4.jpg",
]

LABELS = ['MES 0', 'MES 1', 'MES 2', 'MES 3']

# ==========================================
# 2. LOAD MODELS & DEFINITIONS (Sama persis, tidak ada perubahan)
# ==========================================
def GroupConv2D(filters, kernel_size, strides=(1, 1), padding='same', groups=3):
    def layer(x): return tf.keras.layers.Conv2D(filters, kernel_size, strides=strides, padding=padding)(x)
    return layer
def SE2MaxPooling2D(pool_size=(2, 2)):
    def layer(x): return tf.keras.layers.MaxPooling2D(pool_size=pool_size)(x)
    return layer
def SE2LiftingLayer(x):
    def layer(x): return x
    return layer
def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred): return tf.reduce_mean(y_pred)
    return loss

print("📦 Loading System...")
try:
    custom_objs = {'loss': focal_loss(), 'GroupConv2D': GroupConv2D, 'SE2MaxPooling2D': SE2MaxPooling2D, 'SE2LiftingLayer': SE2LiftingLayer}
    hybrid_model = load_model(os.path.join(MODEL_DIR, "TryFindingBestModel.h5"), custom_objects=custom_objs)
    scaler_20 = joblib.load(os.path.join(MODEL_DIR, "scaler_handcrafted_20.pkl"))
    umap_model = joblib.load(os.path.join(MODEL_DIR, "umap_model_mixed.pkl"))
    lgbm_agent = lgb.Booster(model_file=os.path.join(MODEL_DIR, "feedback_agent.txt"))
    scaler_23 = joblib.load(os.path.join(MODEL_DIR, "scaler_agent.pkl"))
    print("✅ System Loaded Successfully.")
except Exception as e:
    print(f"❌ Error Load: {e}")

# ==========================================
# 3. HELPER FUNCTIONS (Sama persis, tidak ada perubahan)
# ==========================================
def extract_features(img_rgb):
    if len(img_rgb.shape) == 3: gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    else: gray = img_rgb
    coeffs = pywt.dwt2(gray, WAVELET)
    LL, (LH, HL, HH) = coeffs
    def s(sub): return [np.mean(sub), np.std(sub), np.var(sub), scipy.stats.entropy(np.abs(sub.flatten())+1e-6)]
    feats = []
    for b in [LL, LH, HL, HH]: feats.extend(s(b))
    feats.append(np.sum(np.square(HH)))
    g_u8 = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    glcm = graycomatrix(g_u8, [5], [0, np.pi/4, np.pi/2], 256, symmetric=True, normed=True)
    feats += [graycoprops(glcm, p).mean() for p in ['contrast', 'dissimilarity', 'homogeneity']]
    return feats

def get_model_inputs(img_rgb):
    try: cr = img_rgb[30:430, 200:550]
    except: cr = img_rgb
    re = cv2.resize(cr, IMG_SIZE)
    
    ft = extract_features(re)
    ft_s = scaler_20.transform(np.array(ft).reshape(1, -1))
    um = umap_model.transform(ft_s)
    inp = np.expand_dims(re.astype(np.float32)/255.0, axis=0)
    
    return re, inp, ft_s, um

# ==========================================
# 4. CORE HEATMAP GENERATOR (Sama persis, tidak ada perubahan)
# ==========================================
# ==========================================
# 4. CORE HEATMAP GENERATOR (DIPERBAIKI: EFEK GLOW KUNING-MERAH)
# ==========================================
def generate_attention_map(img_path, mode='lgbm', patch_size=35, stride=10):
    img_raw = cv2.imread(img_path)
    if img_raw is None: return None, None, None, 0.0
    img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
    
    re, inp, ft_s, um = get_model_inputs(img_rgb)
    k_pred = hybrid_model.predict([inp, ft_s, um], verbose=0)
    cnn_conf = np.max(k_pred)
    target_idx = np.argmax(k_pred[0]) 
    
    if mode == 'lgbm':
        l_in = np.concatenate(([cnn_conf], um[0], ft_s[0])).reshape(1, -1)
        l_pred = lgbm_agent.predict(scaler_23.transform(l_in))[0]
        if isinstance(l_pred, np.ndarray): 
            base_score = np.max(l_pred)
            target_idx = np.argmax(l_pred)
        else: 
            base_score = l_pred if l_pred > 0.5 else (1-l_pred)
            target_idx = int(l_pred > 0.5)
    else:
        base_score = k_pred[0][target_idx]

    w, h = IMG_SIZE
    heatmap = np.zeros((w, h))
    base_img = re.copy()
    masks, coords = [], []
    
    # --- LOGIC LOOP PATCHING ---
    for y in range(0, h-patch_size+1, stride):
        for x in range(0, w-patch_size+1, stride):
            m = base_img.copy()
            m[y:y+patch_size, x:x+patch_size, :] = 128
            masks.append(m)
            coords.append((x,y))
            
    for i, (x,y) in enumerate(coords):
        masked = masks[i]
        if mode == 'cnn':
            m_inp = np.expand_dims(masked.astype(np.float32)/255.0, axis=0)
            pred = hybrid_model.predict([m_inp, ft_s, um], verbose=0)[0][target_idx]
        else:
            f_new = extract_features(masked)
            fs_new = scaler_20.transform(np.array(f_new).reshape(1, -1))
            u_new = umap_model.transform(fs_new)
            ln_new = np.concatenate(([cnn_conf], u_new[0], fs_new[0])).reshape(1, -1)
            lp_new = lgbm_agent.predict(scaler_23.transform(ln_new))[0]
            if isinstance(lp_new, np.ndarray): pred = lp_new[target_idx]
            else: pred = lp_new if target_idx==1 else (1-lp_new)
        heatmap[y:y+patch_size, x:x+patch_size] = base_score - pred

    # === [VISUALISASI GLOW / PENDARAN CAHAYA] ===
    
    # 1. Bersihkan nilai negatif
    heatmap = np.maximum(heatmap, 0)
    
    # 2. Normalisasi Awal
    if np.max(heatmap) > 0: heatmap /= np.max(heatmap)

    # 3. GLOW EFFECT (PENTING):
    # - Tidak ada pemangkasan (Threshold rendah sekali, cuma buang noise 0.05)
    # - Tidak ada pangkat 2 (biarkan linear agar kuning muncul)
    heatmap[heatmap < 0.05] = 0 

    # 4. SUPER WIDE BLUR:
    # Menggunakan kernel besar (45, 45) agar kotak-kotak itu "meleleh"
    # dan menyatu menjadi pendaran cahaya (merah di tengah, kuning di pinggir).
    heatmap_smooth = cv2.GaussianBlur(heatmap, (45, 45), 0)
    
    # Normalisasi ulang setelah blur agar pusatnya tetap merah pekat
    if np.max(heatmap_smooth) > 0: heatmap_smooth /= np.max(heatmap_smooth)
    
    # 5. PEWARNAAN JET (Spectrum Lengkap: Biru-Hijau-Kuning-Merah)
    hm_u8 = np.uint8(255 * heatmap_smooth)
    hm_c = cv2.applyColorMap(hm_u8, cv2.COLORMAP_JET)
    
    # 6. OVERLAY
    overlay = re.copy()
    
    # Masking: Kita tampilkan hampir semua heatmap kecuali yang benar-benar hitam
    # Ini supaya pinggiran kuning/hijaunya ikut ter-render
    mask = heatmap_smooth > 0.05 
    
    if np.any(mask):
        roi_img = re[mask]
        roi_hm = hm_c[mask]
        
        # Komposisi: 60% Gambar Asli + 40% Heatmap (Warna menyala tapi tekstur terlihat)
        blended = cv2.addWeighted(roi_img, 0.6, roi_hm, 0.4, 0)
        overlay[mask] = blended
    
    return overlay, target_idx, base_score, re

# ==========================================
# 5. MAIN LOOP & PLOTTING DENGAN 6 BARIS & TAMPILAN RAPAT
# ==========================================
num_images = len(TARGET_IMAGES)
print(f"🚀 Processing {num_images} Images...")

results = []
for idx, img_path in enumerate(TARGET_IMAGES):
    filename = os.path.basename(img_path)
    print(f"[{idx+1}/{num_images}] Analyzing: {filename} ...")
    if not os.path.exists(img_path):
        print(f"⚠️ Skipping {filename} (File Not Found)")
        continue
    ov_cnn, _, _, _ = generate_attention_map(img_path, mode='cnn')
    ov_lgbm, pred_cls, conf, orig_img = generate_attention_map(img_path, mode='lgbm')
    results.append({
        'orig': orig_img, 'cnn': ov_cnn, 'lgbm': ov_lgbm,
        'label': LABELS[pred_cls], 'fname': filename
    })

# === PENGATURAN FIGURE AGAR RAPAT ===
# Ubah tinggi per baris lebih kecil agar lebih padat
row_height = 2.5 # Nilai ini diperkecil
fig_height = len(results) * row_height
print("🎨 Creating Final Figure...")

# figsize harus cukup lebar untuk 4 kolom, dan tinggi menyesuaikan jumlah baris
fig, axes = plt.subplots(len(results), 4, figsize=(12, fig_height)) # figsize lebar diperkecil
if len(results) == 1: axes = np.expand_dims(axes, axis=0)

# Loop Plotting Utama (Tanpa Title per Subplot agar Rapat)
for i, res in enumerate(results):
    axes[i, 0].imshow(res['orig'])
    axes[i, 0].axis('off') # Tidak ada axis, tidak ada title

    axes[i, 1].imshow(res['cnn'])
    axes[i, 1].axis('off')

    axes[i, 2].imshow(res['lgbm'])
    axes[i, 2].axis('off')

    axes[i, 3].axis('off')
    # Teks Label dibuat besar dan bold di sebelah kanan
    axes[i, 3].text(0.1, 0.5, res['label'], ha='left', va='center', 
                    fontsize=20, weight='bold', color='black')

# --- BAGIAN COLORBAR DI KANAN (Menggunakan Gambar sebagai Referensi) ---

# 1. Atur layout agar subplot sangat rapat
# wspace dan hspace yang kecil (mendekati 0) membuat jarak antar subplot hilang
plt.subplots_adjust(left=0.05, right=0.88, top=0.98, bottom=0.02, wspace=0.05, hspace=0.05)

# 2. Dapatkan posisi total area gambar (kolom 1, 2, 3) untuk menempatkan colorbar di kanannya
# Ambil posisi dari axis paling kiri atas dan axis kolom ke-3 paling bawah
pos_top_left = axes[0, 0].get_position()
pos_bottom_right_img = axes[-1, 2].get_position() # Kolom ke-2 (indeks 2) adalah kolom gambar terakhir

cbar_height = pos_top_left.ymax - pos_bottom_right_img.ymin
cbar_bottom = pos_bottom_right_img.ymin

# 3. Buat axis manual untuk colorbar di sebelah kanan kolom gambar 'Stage 2'
# Koordinat [left, bottom, width, height]
cbar_ax = fig.add_axes([0.90, cbar_bottom, 0.02, cbar_height])

# 4. Buat mappable untuk colormap 'jet'
norm = mpl.colors.Normalize(vmin=0, vmax=1)
mappable = mpl.cm.ScalarMappable(norm=norm, cmap='jet')

# 5. Gambar colorbar vertikal
cb = fig.colorbar(mappable, cax=cbar_ax, orientation='vertical')
# Tidak perlu label 'Attention Level' agar mirip gambar referensi
# cb.set_label('Attention Level', fontsize=12, labelpad=10) 

# Hapus ticks agar bersih seperti di gambar referensi (opsional)
cb.set_ticks([]) 

save_path = f"Final_Result_visualisation_test.jpg"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"✅ DONE! Saved to {save_path}")
plt.show()